<a href="https://colab.research.google.com/github/josephibekwe/JI/blob/main/EDA_Preprocessing_BotV12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Support Classification

## Problem Understanding

This project looks at **short-text multi-class intent classification** in the customer support domain — given a short customer message, the model must predict which of 27 intents (for example *cancel_order*, *track_refund*, *contact_human_agent*) the customer is expressing.

The task is interesting for three reasons:

- **Short messages contain few useful words.** The median message is only about 9 words long, so there is very little text for the model to work with. Bag-of-words models rely on counting words, and with so few words per message there is not much to count — this is why contextual models, which also use word order and meaning, are more suitable here.
- **Some intents share the same vocabulary.** The 27 intents include pairs like *cancel_order* and *change_order* that use many of the same words. A model cannot separate them by keywords alone — it has to understand what the customer is actually asking for.
- **The data includes different writing styles.** The Bitext dataset labels each message with flags for things like misspellings, colloquial phrasing, and politeness. This means the data covers the kind of variation real customers produce, rather than only clean, well-formed sentences.


### What intent classification means

**Intent classification** = given a piece of text, predict which *intent* (purpose, goal) the writer is expressing, from a fixed list of possible intents.

Breaking that down:

- **Input:** a short piece of text (usually a customer message, a chatbot query, or a voice-assistant command).
- **Output:** one label from a predefined list.
- **The labels are intents** — they describe *what the user wants*, not *what the text is about*.

It is a specific sub-type of text classification. The word *intent* is what makes it intent classification rather than generic text classification.


### Research questions

- **RQ1 — Classical ceiling.** How well can a simple linear classifier with TF-IDF distinguish 27 overlapping classes?
- **RQ2 — Recurrent uplift.** Do sequence models (BiLSTM, GRU) with pretrained GloVe embeddings give a meaningful improvement over the classical baseline on messages this short?
- **RQ3 — Transformer payoff.** Do pretrained transformer- BERT-base, justify their much larger size and training cost on a 27-class, 27k-sample dataset.


### Dataset

The Bitext Customer Support LLM Chatbot Training Dataset is used — 26,872 messages, 27 intent classes grouped into 10 higher-level categories, loaded from HuggingFace. The dataset is synthetic: classes are nearly balanced (~995 messages per class), there are no missing values or duplicates, and each message carries linguistic-variation flags. The sample size comfortably exceeds the ≥6,000 minimum in the brief.

**Environment Setup- GPU Check**
Here we verify if GPU accelration (CUDA) is available in Colab environment.
This step also displays;


*   The memory being used (CPU or GPU),
*   The GPU Name
*   Available RAM memory






In [ ]:
import torch

#Check if CUDA (GPU support) is available
print("CUDA available:", torch.cuda.is_available()) # Check if CUDA is available for GPU acceleration

#Display available GPU name OR default to CPU
print("Device name   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

#Dispaly total available GPU memory (VRAM)
print("VRAM (GB)     :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
      if torch.cuda.is_available() else "n/a")

CUDA available: True
Device name   : Tesla T4
VRAM (GB)     : 15.6


**Environment Setup Package**
This step install python libraries required for this project, whihc include;


*   Deep learning (pytorch, TensorFlow, Transformers)
*   Data processing/anlaysis (pandas, Numpy, SciPy)
*   Machine Learning (Scikit-learn)
*  Model Evaluation (Evaluate, Accelearte), Visalulisation( Plotly), Etc.






In [ ]:
import subprocess, sys

#List of requires libraries for data processing, ML,and deep learning
pkgs = ["torch","transformers","datasets","accelerate","evaluate",
        "tensorflow","scikit-learn","pandas","numpy","plotly",
        "requests","joblib","scipy"]

#Install all packages using pip
subprocess.check_call([sys.executable,"-m","pip","install","-q"]+pkgs)

#print confirmation message
print("All packages installed")

All packages installed


## Import packages required to execute the different deep learning models

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, re, time, requests, io, joblib
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.special import softmax
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

### Help function:
This section defines the reusable helper functions used to improve consistency throughout the notebook thus making it easy to read

*   'section()' formats and prints clear header formats
*   'print_df' gives a standard display of key dataset information like shape and sample rows
*   =×95 dividers with a title) and print_df() (styled pandas)


In [ ]:
# -----------------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------------
def section(title: str):
    # function to print section titles in a consistent format
    print("\n" + "=" * 95)
    print(title)
    print("=" * 95)

def print_df(df: pd.DataFrame, name: str, n: int = 10):
    # function to print a dataframe's name, shape, and first n rows in a consistent format
    print("\n" + "-" * 95)
    print(f"[DATAFRAME] {name} | shape = {df.shape[0]:,} rows × {df.shape[1]:,} cols")
    print("-" * 95)
    print(df.head(n).to_string())

#confirmation that helper functions are ready for use
print("Helpers ready ")

Helpers ready 


**Configuration Settings**
This secton defines the configuration variables used throughout the notebook.

These include;
including the input feature column, target label column, number of classes,

*   'Num_CLASSES' reflect the number of dataset intent categories
*   'RANDOM_STATE' used for reproducing of results
*   'TARGET' and 'FEATURE' specify dependent and independent varaibles
*   'DEVICE' determines whether calculations run on GPU or CPU
*   Colour schemes for plotting are defined for the notebook

In [ ]:
# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------
#Reproducability
RANDOM_STATE = 42

#Dataset Structure
TARGET       = "intent"
FEATURE_COL  = "instruction"

#Model Setup
NUM_CLASSES  = 27

#Device setup (manully or GPU if required)
DEVICE       = "cpu"

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COLORS_BINARY   = {"0": "#2E86AB", "1": "#D1495B"}
COLORS_SEQ      = px.colors.qualitative.Set2
COLORS_MULTI    = px.colors.qualitative.Set3 + px.colors.qualitative.Dark24
C_BLUE, C_RED   = "steelblue", "#D1495B"

#confirmation message
print("configuration ready")

configuration ready


# Dataset

**Dataset:** Bitext Customer Support LLM Chatbot Training Dataset, loaded from HuggingFace (~27,000 samples).

**Target variable (Y):** `intent` — 27 distinct customer intent classes (e.g. cancel_order, get_refund, track_order). This is a multi-class classification problem with 27 possible output labels.

**Input feature (X):** `instruction` — a free-text customer utterance (e.g. "I need to cancel my order"). This is the only input feature used. No structured or numerical features are present in this dataset.

The intent label directly captures what the customer wants to achieve. Predicting it correctly allows a support system to route the customer to the right response or agent automatically.

The raw customer utterance contains all available signal. No metadata (e.g. account history, product type) is available — the model must classify intent from text alone.

**Task type:** Multi-class text classification — 27 mutually exclusive classes, one prediction per utterance.

**Load Data:**
This step loads Bitext customer support data directly from HuggingFace datasets using server API.
*   data is in parquet file format which is more efficient as it bypasses the datasets library (which has DLL issues on Windows with Application Control policies)

*   Output: Downloads the parquet file and prints a preview — 26,872 rows × 5 columns (flags, instruction, category, intent, response).





In [ ]:
# =============================================================================
# LOAD DATA
# =============================================================================
section("Load dataset from HuggingFace")

def load_bitext() -> pd.DataFrame: # loading the Bitext dataset from HuggingFace's parquet API and returning it as a pandas DataFrame through API calls
    api = ("https://datasets-server.huggingface.co/parquet"
           "?dataset=bitext/Bitext-customer-support-llm-chatbot-training-dataset")
    resp = requests.get(api, timeout=30) # make an API request to get the parquet file URL for the Bitext dataset
    resp.raise_for_status() # check if the API request was successful
    url = resp.json()["parquet_files"][0]["url"] # extract the parquet file URL from the API response
    print(f"Downloading parquet: {url}")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return pd.read_parquet(io.BytesIO(r.content))

df = load_bitext()
print_df(df, "Bitext raw")


Load dataset from HuggingFace

-----------------------------------------------------------------------------------------------
[DATAFRAME] Bitext raw | shape = 26,872 rows × 5 cols
-----------------------------------------------------------------------------------------------
   flags                                                   instruction category        intent                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

# Data Preparation

Before any modelling, the raw text is checked, cleaned, and split into training, validation, and test sets.

This step does four things:

- **Inspect the data.** Shape, column types, missing values, and duplicates are checked so later problems are not caused by bad input.
- **Explore the data.** Class distribution, category grouping, message length, linguistic-variation flags, and sample messages are examined to understand what the models will have to deal with.
- **Clean the text.** Messages are lowercased, URLs and special characters are removed, and extra whitespace is stripped. This reduces vocabulary size without losing the words that actually carry intent information.
- **Split the data.** A stratified 70 / 15 / 15 split is used: 70% for training, 15% for validation (hyperparameter tuning, early stopping), and 15% for final test evaluation. Stratification guarantees all 27 classes are proportionally represented in every split.

The test set is only used once, at the end of the project, to produce the final reported metrics for each model.

# Exploration of Data

Before choosing a model, it helps to understand what the data actually looks like. The EDA checks the shape of the class distribution, the length of the messages, the vocabulary overlap between intents, and the type of linguistic variation the models will have to handle. Each finding below has a direct consequence for which models are suitable and how they should be evaluated.

- **Class distribution.** All 27 intents contain between 950 and 1,000 messages (~995 on average, ~3.7% of the dataset each). The data is near-perfectly balanced because it is synthetically generated. This means no class weighting or resampling is needed, and macro F1 is the correct main metric because every class contributes equally.
- **Category grouping.** The 27 intents belong to 10 higher-level categories (ACCOUNT, ORDER, REFUND, and so on). Some categories contain several intents that share vocabulary — ACCOUNT has 6 intents, ORDER has 4, REFUND has 3 — and these are the most likely sources of confusion in the confusion matrix later.
- **Message length.** Messages are short: median ~9 words, mean ~9 words, maximum 16. Short text weakens bag-of-words models because there are fewer discriminating words per message. This is part of the reason deeper contextual models are expected to help.
- **Linguistic variation flags.** Each message carries flags indicating its writing style — *Basic*, *Colloquial*, *Misspelled*, *Politeness*, *Question-like*, *Offensive*. Most messages are Basic, about a third are Question-like, and roughly 18% are Misspelled. The misspelled subset is the most important for robustness: a model that cannot handle typos is not suitable for real customer traffic, where spelling errors are common.
- **Sample messages.** Reading actual examples shows that similar intents share surface vocabulary — *cancel_order* and *change_order* both contain the words *order* and an action verb, and use `{{Order Number}}` template placeholders. This is qualitative evidence for the confusion hypothesis: TF-IDF alone cannot separate these intents, which is what motivates moving to contextual models.
- **Top words per intent.** Looking at the most common words for each intent shows how much vocabulary is shared between related intents. Intents with unique top words (e.g. *create_account*, *get_invoice*) are expected to score high F1 across all models. Intents whose top words overlap with neighbours (e.g. *cancel_order* vs *change_order*) are expected to produce the low-F1 tail.

These findings set the expectations for the modelling section: the data is clean and balanced, the main difficulty is vocabulary overlap between semantically close intents, and the most interesting robustness question is how each model tier handles misspelled messages.

### 1. Dataset Overview
This section provides a high-level summary of the dataset structure and quality

**Checking for:** Shape, column types, missing values, duplicates.

**Findings:** 4 columns — instruction (input), intent (target), category (grouping), flags (linguistic variation). No missing values, no duplicates.

Prints shape, column names, missing-value counts per column, and duplicate row counts.
Output: Confirms 26,872 rows × 5 cols, 0 missing values, 0 duplicates, all columns are object (string) dtype.
Data is clean — no imputation or deduplication is needed. This is expected because Bitext is a synthetically generated dataset.

In [ ]:
# =============================================================================
# DATASET OVERVIEW
# =============================================================================
section("Dataset overview")

#Diplay shaope and rows
print(f"Shape      : {df.shape[0]:,} rows × {df.shape[1]:,} cols")

#List all column names
print(f"Columns    : {list(df.columns)}")

#Check for missing values
print(f"Missing    : {df.isnull().sum().to_dict()}")

#Count duplicates
print(f"Duplicates : {df.duplicated().sum()}")
print()

#Diplay column types
print(df.dtypes.to_string())


Dataset overview
Shape      : 26,872 rows × 5 cols
Columns    : ['flags', 'instruction', 'category', 'intent', 'response']
Missing    : {'flags': 0, 'instruction': 0, 'category': 0, 'intent': 0, 'response': 0}
Duplicates : 0

flags          object
instruction    object
category       object
intent         object
response       object


### 2. Class Distribution

This section the target variable distribution ('intent).

Here we Evaluation:
*   The number of samples per class
*   % distribution across all 27 intent categories
*   Class balance (mean, min, max samples per class)

**Findings:** Dataset is synthetically generated — near-perfect balance of ~1,000 samples per class. No class weighting or resampling required. Macro F1 is the appropriate primary metric as it treats all 27 classes equally.

Counts rows per intent, prints them as a table (count + percentage), and plots a Plotly bar chart.
Output: All 27 intents have between 950 and 1,000 samples (mean 995.3). Each class sits at ~3.7% of the dataset.
Near-perfect synthetic balance means no resampling or class weighting is needed. Macro F1 is the correct primary metric because it treats all 27 classes equally. ROC-AUC macro-OvR is a valid secondary metric under balance.

In [ ]:
# =============================================================================
# TARGET DISTRIBUTION — 27 INTENT CLASSES
# =============================================================================
section("Target distribution (intent classes)")

vc = df[TARGET].value_counts().reset_index() # count the occurrences of each unique value in the target column and reset the index to get a DataFrame
                                             # with 'intent' and 'count' columns
vc.columns = ["intent", "count"]
vc["pct"] = (vc["count"] / len(df) * 100).round(2)

print(vc.to_string(index=False))
print(f"\nUnique intents : {df[TARGET].nunique()}")
print(f"Min per class  : {vc['count'].min()}")
print(f"Max per class  : {vc['count'].max()}")
print(f"Mean per class : {vc['count'].mean():.1f}")

fig = px.bar( # plotting a horizontal bar chart of the intent class distribution using Plotly Express, with bars colored by count and a blue color scale
    vc.sort_values("count"),
    x="count", y="intent",
    orientation="h",
    title="Intent class distribution — all 27 classes",
    color="count",
    color_continuous_scale="Blues",
    labels={"count": "Number of samples", "intent": "Intent"}
)
fig.update_layout(title_x=0.02, height=700, yaxis_title=None)
fig.show()


Target distribution (intent classes)
                  intent  count  pct
contact_customer_service   1000 3.72
               complaint   1000 3.72
           check_invoice   1000 3.72
          switch_account   1000 3.72
            edit_account   1000 3.72
     contact_human_agent    999 3.72
   check_payment_methods    999 3.72
         delivery_period    999 3.72
 newsletter_subscription    999 3.72
             get_invoice    999 3.72
           payment_issue    999 3.72
   registration_problems    999 3.72
            cancel_order    998 3.71
             place_order    998 3.71
            track_refund    998 3.71
            change_order    997 3.71
 set_up_shipping_address    997 3.71
     check_refund_policy    997 3.71
          create_account    997 3.71
              get_refund    997 3.71
                  review    997 3.71
        delivery_options    995 3.70
          delete_account    995 3.70
        recover_password    995 3.70
             track_order    995 3.70


### 3. Category Distribution
**Checking for:** How the 27 intents group into 10 high-level categories.

**Model signals:** Categories with multiple similar intents (e.g. ORDER contains cancel_order and change_order) will produce the most confusion — confirmed in the confusion matrix later.

Groups the 27 intents into their high-level category column (ACCOUNT, ORDER, REFUND, etc.) and prints counts.
11 categories: ACCOUNT (5,986) , CANCEL (950). Heavy skew at the category level because ACCOUNT contains 6 intents.

In [ ]:
# =============================================================================
# CATEGORY DISTRIBUTION (10 HIGH-LEVEL GROUPS)
# =============================================================================
section("Category distribution")

cat_vc = df["category"].value_counts().reset_index() # count the occurrences of each unique value in the 'category' column and reset the index to get a DataFrame with 'category' and 'count' columns
cat_vc.columns = ["category", "count"]

fig = px.bar(
    cat_vc,
    x="category", y="count",
    title="Sample count by category (high-level groups)",
    color="category",
    color_discrete_sequence=COLORS_SEQ,
    labels={"count": "Count", "category": "Category"}
)
fig.update_layout(title_x=0.02, showlegend=False, xaxis_tickangle=-30)
fig.show()

print(cat_vc.to_string(index=False))


Category distribution


    category  count
     ACCOUNT   5986
       ORDER   3988
      REFUND   2992
     CONTACT   1999
     INVOICE   1999
     PAYMENT   1998
    FEEDBACK   1997
    DELIVERY   1994
    SHIPPING   1970
SUBSCRIPTION    999
      CANCEL    950


### 4. Text Length Distribution
**Checking for:** Utterance length in words and characters.

**Model signals:** Median ~8 words. Short text weakens bag-of-words models as there are fewer discriminating terms per sample. Transformer models handle short sequences natively without truncation.

Computes n_chars and n_words per utterance, prints summary statistics, and plots the distribution.
Output: Mean 8.69 words / 46.89 chars, median 9, max 16. Very short utterances.
Why it matters: With a median of just 9 words, bag-of-words models have few discriminating terms per sample — this caps what TF-IDF can achieve. Transformer models handle short sequences natively, so this favours the transformer tier over classical ML.

In [ ]:
# =============================================================================
# TEXT LENGTH ANALYSIS
# =============================================================================
section("Text length analysis")

df["n_chars"] = df[FEATURE_COL].str.len() # # checking the length of the text in characters by applying the string length function to the feature column and storing it in a new column 'n_chars'
df["n_words"] = df[FEATURE_COL].str.split().str.len() # checking the length of the text in words by splitting the text on whitespace and counting the resulting list's length, storing it in a new column 'n_words'

print("-" * 80)
print(df[["n_chars", "n_words"]].describe().round(2).to_string())

# creating histograms to visualise the distribution of character lengths and word counts in the instruction texts, and a box plot to show the word count distribution by category
fig = px.histogram(
    df, x="n_chars", nbins=60,
    title="Instruction — character length distribution",
    labels={"n_chars": "Character length", "count": "Count"}
)
fig.update_traces(marker_color=C_BLUE)
fig.update_layout(bargap=0.05, title_x=0.02)
fig.show()

fig = px.histogram(
    df, x="n_words", nbins=40,
    title="Instruction — word count distribution",
    labels={"n_words": "Word count", "count": "Count"}
)
fig.update_traces(marker_color=C_RED)
fig.update_layout(bargap=0.05, title_x=0.02)
fig.show()

fig = px.box( # creating a box plot to show the distribution of word counts by category, with boxes colored by category and using a predefined color sequence
    df, y="n_words", x="category",
    title="Word count by category",
    color="category",
    color_discrete_sequence=COLORS_SEQ
)
fig.update_layout(title_x=0.02, xaxis_tickangle=-30, showlegend=False)
fig.show()


Text length analysis
--------------------------------------------------------------------------------
        n_chars   n_words
count  26872.00  26872.00
mean      46.89      8.69
std       10.90      2.61
min        6.00      1.00
25%       40.00      7.00
50%       48.00      9.00
75%       55.00     11.00
max       92.00     16.00


### Intent density per category

This analysis counts how many distinct intents live inside each high-level category.

**Checking for:** Which categories contain the most intents — i.e. which categories will be most competitive for the model to separate.

**Findings:** ACCOUNT holds 6 intents, ORDER holds 4, REFUND holds 3 — all others have 1–2. These three crowded categories are the predicted confusion hotspots, because intents in the same category share vocabulary and action verbs (e.g. ORDER contains *cancel_order*, *change_order*, *delivery_options*, *track_order* — all share the word "order" plus an action verb).

**Model signals:** The confusion matrix later should show that most off-diagonal errors concentrate inside ACCOUNT, ORDER, and REFUND, and that cross-category errors are rare.

In [ ]:
# =============================================================================
# INTENT DENSITY BY CATEGORY
# Count how many distinct intents live inside each high-level category.
# Categories with many intents are the predicted confusion hotspots because
# intents in the same category share vocabulary (e.g. ORDER contains
# cancel_order, change_order, delivery_options, track_order — all of which
# share the word "order" and an action verb).
# =============================================================================
section("Intent density per category")

intent_cat = (
    df.groupby("category")["intent"]
      .nunique()
      .reset_index()
      .rename(columns={"intent": "unique_intents"})
      .sort_values("unique_intents", ascending=False)
)

styled_table = (
    intent_cat.style
    .background_gradient(subset=["unique_intents"], cmap="Blues")
    .set_properties(**{
        "background-color": "white",
        "color": "black",
        "border-color": "black"
    })
)
styled_table


Intent density per category


,category,unique_intents
0,ACCOUNT,6
6,ORDER,4
8,REFUND,3
5,INVOICE,2
2,CONTACT,2
4,FEEDBACK,2
3,DELIVERY,2
9,SHIPPING,2
7,PAYMENT,2
1,CANCEL,1


### 6. Sample Utterances
**Checking for:** Vocabulary overlap between similar intents.

**Key finding:** cancel_order and change_order share keywords such as order, cancel, change. This overlap is the primary source of confusion matrix off-diagonal entries and appears across all model tiers.

Prints 3 raw example utterances each from a set of chosen intents (cancel_order, track_order, etc.).
Output: Examples showing {{Order Number}} template placeholders and real vocabulary overlap — for example, both cancel_order and change_order contain the words "order" and action verbs.
This is qualitative evidence for the confusion hypothesis — intents in the same category genuinely share keywords, so TF-IDF alone cannot separate them. This motivates moving beyond classical ML to contextual models.

In [ ]:
# =============================================================================
# SAMPLE UTTERANCES PER INTENT
# =============================================================================
section("Sample utterances per intent (3 examples each)")

sample_intents = ["cancel_order", "track_order", "get_refund",
                  "contact_human_agent", "complaint"]
for intent_name in sample_intents: # function to print 3 random sample utterances for each of the specified intent names,
                                    # with a header showing the intent name in uppercase and a bullet point for each sample
    print("\n" + "-" * 80)
    print(f"{intent_name.upper()}")
    samples = df[df[TARGET] == intent_name][FEATURE_COL].sample(3, random_state=42).tolist()
    for s in samples:
        print(f"  • {s}")


Sample utterances per intent (3 examples each)

--------------------------------------------------------------------------------
CANCEL_ORDER
  • I cannot afford purchase {{Order Number}}
  • I have bought some item, I have to cancel order {{Order Number}}
  • I do not want this item, cancel order {{Order Number}}

--------------------------------------------------------------------------------
TRACK_ORDER
  • where to see the bloody ETA of the order {{Order Number}}
  • checking status of order {{Order Number}}
  • see purchase {{Order Number}} current status

--------------------------------------------------------------------------------
GET_REFUND
  • I paid {{Currency Symbol}}{{Refund Amount}} for this product, I need to get a rebate
  • I am trying to receive a compensation
  • i need assistance demanding compensations of my money

--------------------------------------------------------------------------------
CONTACT_HUMAN_AGENT
  • I can't speak iwth an operator
  • I want he

### 7. Top Words per Intent
**Checking for:** Whether each intent has a sufficiently distinct vocabulary for bag-of-words separation.

**Model signals:** Intents with unique top words achieve high F1 across all models. Intents sharing top words with neighbours produce low F1 — particularly for TF-IDF-based classical ML.

Uses collections.Counter to compute the most frequent words for the three largest intents.
Output: Empty output — only the section header prints. The table is computed but never displayed.
This is a rendering bug to fix — likely a missing print(...) or display(...) at the end of the cell. The analysis is intended to show whether each intent has a discriminative top-N vocabulary.

In [ ]:
# =============================================================================
# TOP WORDS PER INTENT — TERM FREQUENCY OVERVIEW
# =============================================================================
section("Most common words per intent (top 3 intents by size)")

from collections import Counter

# For the top 3 intents by sample size, this code extracts the instruction texts,
# tokenizes them into words, removes common stopwords and short words, counts the frequency of each remaining word,
# and visualizes the top 10 most common words for each intent using horizontal bar charts colored by frequency.

top_intents = df[TARGET].value_counts().head(3).index.tolist()
for intent_name in top_intents:
    subset = df[df[TARGET] == intent_name][FEATURE_COL]
    words = " ".join(subset).lower().split()
    stopwords = {"i", "my", "to", "the", "a", "an", "is", "it", "me", "can", "you"}
    words = [w for w in words if w not in stopwords and len(w) > 2]
    common = Counter(words).most_common(10)
    common_df = pd.DataFrame(common, columns=["word", "freq"])

    fig = px.bar(
        common_df, x="freq", y="word", orientation="h",
        title=f"Top words — {intent_name}",
        color="freq",
        color_continuous_scale="Blues",
        labels={"freq": "Frequency", "word": "Word"}
    )
    fig.update_layout(title_x=0.02, yaxis={"categoryorder": "total ascending"},
                      showlegend=False, height=350)
    fig.show()


Most common words per intent (top 3 intents by size)


## Feature Engineering & Representation

Text cannot be fed directly into any model — it must first be converted to numbers.
Three different representations are built for the two model tiers.

- **TF-IDF (for classical ML).** Each message is turned into a sparse numerical vector that records how informative each word (or two-word phrase) is for that message relative to the rest of the corpus. Unigrams and bigrams are used, the vocabulary is capped at 50,000 features, and the vectoriser is fitted on the training set only to avoid leakage.
- **Padded integer sequences + GloVe embeddings (for BiLSTM and GRU).** Each message is tokenised into integer IDs and padded to a fixed length of 50 tokens. The embedding layer is initialised with pretrained 100-dimensional GloVe vectors so the model starts with general word meaning learned from 6 billion words of text, rather than having to learn it from scratch on the small training set.
- **WordPiece tokenisation (for BERT).** The transformer's own pretrained tokeniser is used, which splits rare words into known sub-word pieces (for example *cancellations* becomes `cancel` + `##lations`). This must be the checkpoint's own tokeniser — using any other would break the pretrained embedding layer.

Labels are encoded to integers 0–26 with `LabelEncoder`, fitted on the full label set so the mapping is the same across every split and every model.

### Text Cleaning

**Why clean text before modelling?** Raw customer utterances contain noise, mixed case, punctuation, URLs — that adds vocabulary size without adding discriminative signal. Reducing noise improves model generalisation.

**Decisions made and why:**

| Decision | Justification |
|----------|---------------|
| Lowercase | "Cancel" and "cancel" are the same word. Lowercasing reduces vocabulary size without losing meaning |
| Remove URLs | No URLs appear in customer utterances but included as a defensive step |
| Remove special characters | Punctuation (!, ?, .) adds noise without adding intent signal for classification |
| Normalise whitespace | Prevents tokenisation errors from extra spaces or newlines |
| Stemming / lemmatisation | TF-IDF with bigrams preserves enough morphological information. Stemming risks collapsing intent-relevant distinctions — e.g. "ordering" vs "ordered" |
| Stop word removal | Short utterances have few words — removing stop words could discard useful context (e.g. "I can't" vs "I can") |

In [ ]:
# =============================================================================
# TEXT CLEANING
# =============================================================================
section("Text cleaning")

def clean_text(t): # function to clean text by converting it to lowercase, removing URLs, special characters, and extra whitespace
    t = str(t).lower()
    t = re.sub(r"http\S+", "", t)          # remove URLs
    t = re.sub(r"[^a-z0-9\s]", " ", t)    # remove special chars
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["text_clean"] = df[FEATURE_COL].apply(clean_text)

# Show before / after
print("-" * 80)
for i in range(3):
    print(f"RAW  : {df[FEATURE_COL].iloc[i]}")
    print(f"CLEAN: {df['text_clean'].iloc[i]}")
    print()


Text cleaning
--------------------------------------------------------------------------------
RAW  : question about cancelling order {{Order Number}}
CLEAN: question about cancelling order order number

RAW  : i have a question about cancelling oorder {{Order Number}}
CLEAN: i have a question about cancelling oorder order number

RAW  : i need help cancelling puchase {{Order Number}}
CLEAN: i need help cancelling puchase order number



### 9. Label Encoding

**Why:** Models require numerical labels, not strings. The intent column contains 27 text class names converted to integers 0–26.

**Method:** LabelEncoder — alphabetical ordering. Same encoder object reused across all notebooks for consistent label-to-intent mapping.

**Leakage check** — fitted on full label set (all 27 classes known upfront), no data leakage risk.


This step converts the target variable ('intent') from catgroical labels to numerical values required for the ML models' input;
*   it assigns unique integer to each inent class and,
*   maintains a consistent  mapping between labels and original intent class
the number of clsses is also updated to reflect the encoded dataset and the mapping table is displays so the encoded labels can be intepreted.



In [ ]:
# =============================================================================
# LABEL ENCODING
# =============================================================================
section("Label encoding")

from sklearn.preprocessing import LabelEncoder
# using label encoding to convert the target variable (intent) into numerical labels
# that can be used for machine learning models, and storing the number of unique classes in NUM_CLASSES

#initialise label encode
le = LabelEncoder()

#convert target abels into numerical formal
df["label"] = le.fit_transform(df[TARGET])

#update number of classes after encoding
NUM_CLASSES = len(le.classes_)

#Display number of classes
print(f"Classes : {NUM_CLASSES}")

#show mapping of numerical labels to original intent names
print("\nIntent → Label mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i:2d} → {cls}")


Label encoding
Classes : 27

Intent → Label mapping:
   0 → cancel_order
   1 → change_order
   2 → change_shipping_address
   3 → check_cancellation_fee
   4 → check_invoice
   5 → check_payment_methods
   6 → check_refund_policy
   7 → complaint
   8 → contact_customer_service
   9 → contact_human_agent
  10 → create_account
  11 → delete_account
  12 → delivery_options
  13 → delivery_period
  14 → edit_account
  15 → get_invoice
  16 → get_refund
  17 → newsletter_subscription
  18 → payment_issue
  19 → place_order
  20 → recover_password
  21 → registration_problems
  22 → review
  23 → set_up_shipping_address
  24 → switch_account
  25 → track_order
  26 → track_refund


### 10. Train / Validation / Test Split — 70 / 15 / 15

This step splits the dataset into training, validation, and test sets using 70/15/15 ratio.

**split the data** To evaluate whether a model generalises to unseen examples. Training and evaluating on the same data produces misleadingly high metrics — the model may have simply memorised the training examples.

**Split rationale:**

| Split | Size | Purpose |
|-------|------|---------|
| Train (70%) | ~18,900 samples | Model training — ~700 examples per class |
| Validation (15%) | ~4,050 samples | Hyperparameter tuning and early stopping — never used for final metrics |
| Test (15%) | ~4,050 samples | Final evaluation — used exactly once per model, never during training |

**stratified:** With 27 classes, a random split risks some splits receiving zero samples for a particular intent. Stratification guarantees all 27 classes are proportionally represented in every split — essential for reliable evaluation.

**70/15/15:** 70% training provides ~700 samples per class — sufficient for classical ML and marginal for deep learning. Splitting the remaining 30% equally between validation and test gives balanced tuning and evaluation sets.

In [ ]:
# =============================================================================
# STRATIFIED TRAIN / VALIDATION / TEST SPLIT  70 / 15 / 15
# =============================================================================
section("Train / val / test split")

from sklearn.model_selection import train_test_split

# splitting the dataset into training, validation, and test sets using stratified sampling
# to maintain the same class distribution across all sets, with 70% of the data for training, 15% for validation, and 15% for testing

#define feature and target
X = df["text_clean"].values
y = df["label"].values

#split 1: seperate test set(15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE)

#Spli2: splits remaining dataset into train (70%) and validation(15%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=RANDOM_STATE)

print(f"Train : {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val   : {len(X_val):,} samples  ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test  : {len(X_test):,} samples  ({len(X_test)/len(X)*100:.1f}%)")
print(f"Total : {len(X):,}")
print(f"\nClasses in train: {len(np.unique(y_train))} | val: {len(np.unique(y_val))} | test: {len(np.unique(y_test))}")


Train / val / test split
Train : 18,809 samples (70.0%)
Val   : 4,032 samples  (15.0%)
Test  : 4,031 samples  (15.0%)
Total : 26,872

Classes in train: 27 | val: 27 | test: 27


---
### Feature Engineering — Deep Learning (BiLSTM, GRU)

Deep learning models (BiLSTM, GRU) require text converted to padded integer sequences fed through an embedding layer — not raw text strings.

**Why pretrained GloVe instead of random embeddings?**
With only ~700 training samples per class, the model does not have enough data to learn word meanings from scratch. GloVe vectors were pretrained on 6 billion words — they already know that "cancel" and "cancellation" are related before training even starts. The notebook will still run without GloVe (falls back to random initialisation) but results will be weaker.

**Two conditions tested for BiLSTM:**
- Frozen — GloVe weights fixed, pretrained knowledge preserved
- Fine-tuned — weights updated during training, adapts to customer support domain

**Section: Tokenisation and padding.** This step prepares the data for Deep Leanring models
*   **Tokenisation converts**  text sequences into interger indices
*   **Padding and Truncation**  ensures al sequcnes have fixed length
*   **Vocabularisation limi**t restricts top 30,00 words to reduce complexity

The maximum sequence length is chosen based on EDA- 50 tokens account for 95% of all text inputs

Fits a Keras Tokeniser on the training text only, converts all three splits to integer sequences, and pads them to MAX_LEN=50.
Output: Vocabulary size 2,368 words, X_train_seq shape (18809, 50). max_len=50 chosen to cover the 95th percentile of utterance lengths from EDA.

Tokenisation is fiited  **only on the training ** data to prevent leekage.

The small vocabulary (2,368) reflects the narrow customer-support domain — useful because the model has less to learn, but also limits generalisation to out-of-domain text.

In [ ]:
# =============================================================================
# FEATURE ENGINEERING — Deep Learning: BiLSTM, GRU
# Tokenisation + padding + GloVe embedding matrix
# =============================================================================
section("Tokenisation & padding Deep Learning")

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

#Define vocabulary size, sequence length, and embedding dimension
MAX_VOCAB = 30_000
MAX_LEN   = 50
EMBED_DIM = 100

#initialise tokenizer with vocabulary limit and 00V handling
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")

#Fit tokenizer into training data only (to avoid leakage)
tokenizer.fit_on_texts(X_train)  # fit on train only

#convert text sequence to sequcnes and apply padding
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train),
                             maxlen=MAX_LEN, padding="post", truncating="post")
X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val),
                             maxlen=MAX_LEN, padding="post", truncating="post")
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),
                             maxlen=MAX_LEN, padding="post", truncating="post")

#displaying key information about data
print(f"Vocabulary size  : {len(tokenizer.word_index):,}")
print(f"X_train_seq shape: {X_train_seq.shape}")
print(f"max_len=50 covers 95th percentile of utterance lengths from EDA")
print("\n Used by: NB3 (BiLSTM, GRU)")


Tokenisation & padding Deep Learning
Vocabulary size  : 2,368
X_train_seq shape: (18809, 50)
max_len=50 covers 95th percentile of utterance lengths from EDA

 Used by: NB3 (BiLSTM, GRU)


### Why GloVe Embeddings?

BiLSTM and GRU cannot read text — they need numbers. Each word is first
converted to an integer index (e.g. "cancel" → 452). The embedding layer
then maps that index to a dense vector of 100 numbers that the model can
learn from.

The simplest approach would be to start with random vectors and let the
model figure out word meanings during training. The problem is that with
only ~700 training samples per intent class, there is not enough data to
learn what words mean from scratch — the model would not converge well.

**GloVe provides pretrained vectors** where similar words are already close
together in vector space, learned from 6 billion words (Wikipedia +
Gigaword) before training even starts:

- "cancel" and "cancellation" → similar vectors
- "refund" and "money back" → similar vectors
- "cancel" and "invoice" → very different vectors

This gives the model a semantic head start instead of starting from zero.

**GloVe is not strictly required** — the notebook will still run without it.
If the GloVe file is not downloaded, the code automatically falls back to
random initialisation and BiLSTM and GRU will still train. Results will
be a few percentage points lower but nothing will break.

**Two conditions are tested:**

| Condition | Embedding weights | What it tests |
|-----------|------------------|---------------|
| Frozen | Fixed — pretrained GloVe knowledge preserved | Is pretrained knowledge alone sufficient? |
| Fine-tuned | Updated during training | Does adapting GloVe to customer support language improve F1? |

Fine-tuned is expected to outperform frozen — GloVe was trained on
Wikipedia, not customer support conversations. Allowing the weights to
adjust gives the model a head start AND domain adaptation.

This step integrates the pretrained GloVE embedding (Global Vectors for Word Representation) into the model:

*   it loads pretrained word vestors from the GloVe dataset
*   it maps tokenisation vocabulary into thier corresponding GloVe vectors
*   it constructs and embedding matrix for the tokenized indexes

In [ ]:
# =============================================================================
# GLOVE EMBEDDING MATRIX
# Download: wget http://nlp.stanford.edu/data/glove.6B.zip && unzip glove.6B.zip
# =============================================================================

# can be used for BiLSTM and GRU

section("GloVe embedding matrix — BiLSTM / GRU")

#Path to Pre-trained Glove embedding
GLOVE_PATH = "/content/glove.6B.100d.txt"

#ensure glove embedding fits  before proceeding
assert os.path.exists(GLOVE_PATH), f"GloVe file not found at: {GLOVE_PATH}"

#Load Glove embedding with dictionary: {word-to-vector}
embeddings_index = {}
with open(GLOVE_PATH, encoding="utf-8") as f:
  for line in f:
    values = line.split()
    embeddings_index[values[0]] = np.asarray(values[1:], dtype="float32")

print(f"GloVe vectors loaded: {len(embeddings_index):,}")

#Define vocabulary size
vocab_size = min(MAX_VOCAB, len(tokenizer.word_index)) + 1

# Populate embdding  matrix with Glove values
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))
found = 0
for word, idx in tokenizer.word_index.items():
    if idx < vocab_size:
      # Look up the current word in the GloVe embeddings dictionary
      # If the word exists, vec will be its pretrained vector
      # If the word does not exist, vec will be None
      vec = embeddings_index.get(word)
      if vec is not None:
        embedding_matrix[idx] = vec
        found += 1

#calculate ou-of-vocabulary (OOV)
oov = 1 - found / (vocab_size - 1)

#Display embedding matrix details
print(f"Embedding matrix : {embedding_matrix.shape}")
print(f"OOV rate         : {oov:.2%}")

#refernce downstream usage
print("\nUsed by: BiLSTM, GRU (frozen and fine-tuned conditions)")


GloVe embedding matrix — BiLSTM / GRU


AssertionError: GloVe file not found at: /content/glove.6B.100d.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Shared Evaluation Helper

`get_metrics()` is defined here once so all three model notebooks use
the exact same function and return results in the same format.
This ensures we can combine results from all models without any
manual editing.

Returns: `accuracy | precision | recall | f1 | roc_auc`

**Note for LinearSVC:** LinearSVC has no `predict_proba` — it uses
`decision_function` scores instead. These do not sum to 1 so they cannot
be passed directly to `roc_auc_score`. The function detects this
automatically and applies softmax to convert them to pseudo-probabilities
before computing ROC-AUC.

This step defines the evaulation functions to be used across all models
*   Accuracy
*   Macro Precision
*   Macro recall
*   Macreo F1 score
*   Macro ROC_AUC

shares helper functions ensure that there is consistency and fairness in evaluation for all models.






In [ ]:
# =============================================================================
# SHARED EVALUATION HELPER — used by all four models
# =============================================================================

skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

#Dictionary to store models
results = {}

def get_metrics(y_true, y_pred, y_proba,
                model_name=None, tier=None, time_s=None):
    """
    Shared evaluation function — works for all model types.
    model_name, tier, time_s are.
    """

    #convert scores to probability
    scores = np.array(y_proba)
    if not np.allclose(scores.sum(axis=1), 1.0, atol=1e-3):
        scores = softmax(scores, axis=1)

    #calculate evaluation metrics
    m = {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall":    recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1":        f1_score(y_true, y_pred, average="macro"),
        "roc_auc":   roc_auc_score(y_true, scores, multi_class="ovr", average="macro")
    }

    # Add optional fields if provided
    if model_name is not None: m["model"]  = model_name
    if tier       is not None: m["tier"]   = tier
    if time_s     is not None: m["time_s"] = round(time_s, 1)

    # Print results
    label = model_name if model_name else "Model"
    print(f"\n{label} — Test results:")
    for k,v in m.items():
        if k not in ["model","tier","time_s"]:
            print(f"  {k:12}: {v:.6f}")

    return m

def plot_history(history, model_name):
    #plot training and evaluation loss curves for deep learning models
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=history.history["loss"],     name="Train loss",
                             line=dict(color=C_BLUE)))
    fig.add_trace(go.Scatter(y=history.history["val_loss"], name="Val loss",
                             line=dict(color=C_RED, dash="dash")))
    fig.update_layout(title=f"{model_name} — Loss curves",
                      title_x=0.02, xaxis_title="Epoch", yaxis_title="Loss")
    fig.show()

print("get_metrics() ready — accuracy | precision | recall | f1 | roc_auc")

---
# Feature Engineering — Classical ML (Logistic Regression, LinearSVC)

**Representation: TF-IDF sparse matrix**

TF-IDF converts text to a numerical matrix where each value represents how informative a word is for a document relative to the corpus. Works directly with linear classifiers without dense embedding overhead.

| Parameter | Value | Why |
|-----------|-------|-----|
| ngram_range | (1,2) | Bigrams capture two-word phrases - cancel order, track refund |
| max_features | 50,000 | Caps vocabulary for memory efficiency |
| sublinear_tf | True | log(1+tf) - prevents high-frequency words dominating |
| min_df | 2 | Removes rare typos |

**Limitation:** Discards word order entirely. Motivates sequence models in NB3.

This step transforms cleaned text into numerical features TF-IDF (Text Frequency-Invenrse Document Frequency)



*   **Unigrams and Bigrams:** accounts for single word and shor-word phrases
*   **Max feature (50,000)** controls dimension
*   **Sublinear TF** scaling to prevents over dominance of most frequently occuring words
*   **min_df=2** removes rare words ro prevent noise





In [ ]:
# =============================================================================
# FEATURE ENGINEERING — Classical ML: LR, LinearSVC
# TF-IDF sparse matrix — fitted on train, transform applied to val/test
# =============================================================================
section("TF-IDF vectorisation — for Classical ML")

from sklearn.feature_extraction.text import TfidfVectorizer

# Create a TF-IDF vectorizer object
# This converts text into numerical features based on term frequency and inverse document frequenc
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
    min_df=2,
    strip_accents="unicode"
)

#fit training data only, then tranforms the splits
X_train_tfidf = tfidf.fit_transform(X_train)  # fit + transform on train
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

#displays feature space details
print(f"Vocabulary size  : {len(tfidf.vocabulary_):,} features")
print(f"X_train shape    : {X_train_tfidf.shape}")

#calculates sparsity of matrix
print(f"Matrix sparsity  : {1 - X_train_tfidf.nnz/(X_train_tfidf.shape[0]*X_train_tfidf.shape[1]):.4f}")

#references downstream models
print("\nUsed by: NB2 (Logistic Regression, LinearSVC)")

# Model Development

Three tiers of models are implemented so performance can be compared across fundamentally different architectures:

- **Classical ML (Tier 1).** Logistic Regression and LinearSVC are trained on the TF-IDF features. These establish the baseline — if a linear model over TF-IDF already performs well, the problem is mostly about lexical keywords and deeper models will only add marginal value.
- **Recurrent deep learning (Tier 2).** BiLSTM and GRU are trained on padded integer sequences with pretrained GloVe embeddings. Both models are tuned over six configurations each, testing model size, dropout, learning rate, and frozen vs fine-tuned embeddings. A BiLSTM reads the sentence in both directions, while a GRU is a lighter single-direction alternative.
- **Transformers (Tier 3).** DistilBERT and BERT-base-uncased are fine-tuned with their own WordPiece tokenisers. Four configurations are tested, varying the checkpoint, learning rate, and dropout. These models use self-attention, so every word in the message can look at every other word directly — a more expressive representation than the sequential hidden state of an RNN.

Each model is trained on the same training split, tuned on the same validation split, and evaluated on the same test split. Results are recorded through a single shared `get_metrics()` function so the comparison in Step 6 is fair.

### Model 1 — Logistic Regression

**Type:** Linear probabilistic classifier

**How it works:** Models the probability of each of the 27 intent classes directly. Finds a linear decision boundary in TF-IDF feature space that maximises the likelihood of the correct class labels.

**Why chosen:** Provides well-calibrated probability scores — useful for comparing confidence across classes. Good baseline for any text classification task.

This step implements Logistics regression as a baseline Classial Machine leanring model.
*   It  performs 5-fold startified cross-validation of training set using macro F1 score.
*   It trains the model on full training dataset
Results:
*   Cross validation perfromance (mean and standard deviation)
*   Test set evaluation metrics
*   Total training time

This classical model serves as a benchmark for comparing more complex models









In [ ]:
# =============================================================================
# MODEL A — LOGISTIC REGRESSION
# =============================================================================
section("Logistic Regression")

# Record the current time before training starts to calculate the total runtime for this model
t0 = time.time()

# Create a Logistic Regression model object and store it in the variable `lr`
# max_iter=1000      -> allows the optimiser more iterations to converge
# random_state=RANDOM_STATE -> ensures reproducibility
# solver="saga"      -> optimisation algorithm suitable for large / sparse data like TF-IDF
# n_jobs=-1          -> use all available CPU cores where supported
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, solver="saga", n_jobs=-1)

# Perform cross-validation on the training data
# lr              -> the Logistic Regression model to evaluate
# X_train_tfidf   -> TF-IDF feature matrix for the training set
# y_train         -> true labels for the training data
# cv=skf          -> use the predefined Stratified K-Fold splitter
# scoring="f1_macro" -> evaluate using macro F1-score
# n_jobs=-1       -> run cross-validation in parallel using all CPU cores
# The result is an array of F1 scores, one for each fold
cv_lr = cross_val_score(lr, X_train_tfidf, y_train, cv=skf, scoring="f1_macro", n_jobs=-1)
print(f"CV Macro F1 : {cv_lr.mean():.4f} ± {cv_lr.std():.4f}")

# Train the Logistic Regression model on the full training dataset
# The model learns the relationship between TF-IDF features and target labels
lr.fit(X_train_tfidf, y_train)

# Use the trained model to predict class labels for the test set
# Output: predicted intent/department label for each test example
y_pred_lr   = lr.predict(X_test_tfidf)

# Use the trained model to predict class probabilities for the test set
# Output: probability distribution across all classes for each test example
y_proba_lr  = lr.predict_proba(X_test_tfidf)

# Compute evaluation metrics using the shared helper function
# y_test     -> true labels for the test set
# y_pred_lr  -> predicted class labels from Logistic Regression
# y_proba_lr -> predicted class probabilities
# The returned dictionary usually contains metrics like accuracy, precision, recall, F1, and ROC-AUC
lr_metrics = get_metrics(y_test, y_pred_lr, y_proba_lr)

# Store the Logistic Regression results in the shared `results` dictionary
# Key: "Logistic Regression"
# Values stored:
# - "CV F1"   -> mean cross-validation macro F1 score
# - "CV Std"  -> standard deviation of cross-validation score
# - "Time(s)" -> total runtime in seconds since t0
# - **lr_metrics -> unpack all test-set evaluation metrics into the same dictionary
results["Logistic Regression"] = {"CV F1": cv_lr.mean(), "CV Std": cv_lr.std(),
                                   "Time(s)": round(time.time() - t0, 1), **lr_metrics}
print(pd.Series(lr_metrics).round(4).to_string())

### Model 2 — LinearSVC

**Type:** Linear maximum-margin classifier

Finds the hyperplane in TF-IDF feature space that maximises the margin between classes — the decision boundary is as far as possible from all training examples. Typically outperforms Logistic Regression on sparse high-dimensional text.

**Why chosen:** LinearSVC is the standard benchmark for text classification with TF-IDF. Expected to be the strongest classical model.

**cross-validation** Cross-validation was performed on the training set to obtain a more reliable estimate of model performance during model selection. It reduces dependence on a single split, supports hyperparameter tuning, and helps preserve the test set for final unbiased evaluation.

This section implements a Linear SVC (support vector for multi-class classification).


*   It  performs 5-fold startified cross-validation of training set using macro F1 score.
*   It trains the model on full training dataset
*   It evaluates performance of test set
Unlike logistics Regression, Linear SVC uses precision scores (ROC-AUC) for evaluation - not output probabilities.
Linear SVC is a strong competitor to Logistics Regression in text classification



In [ ]:
# =============================================================================
# MODEL B — LINEAR SVC
# =============================================================================
section("LinearSVC")
# Record the current time before training starts to calculate the total runtime for this model
t0 = time.time()

# Create a Linear Support Vector Classifier model and store it in `svc`
# max_iter=2000 -> allows more optimisation iterations for convergence
# random_state=RANDOM_STATE -> ensures reproducibility
# C=1.0-> regularisation parameter; controls the trade-off btw maximising the margin and minimising classification errors
svc = LinearSVC(max_iter=2000, random_state=RANDOM_STATE, C=1.0)

# Perform cross-validation on the training set
# svc            -> the LinearSVC model being evaluated
# X_train_tfidf  -> TF-IDF feature matrix for the training data
# y_train        -> true labels for the training data
# cv=skf         -> use the predefined Stratified K-Fold splitter
# scoring="f1_macro" -> evaluate each fold using macro F1-score
# n_jobs=-1      -> use all available CPU cores for parallel processing
# The result is an array of cross-validation scores, one score per fold
cv_svc = cross_val_score(svc, X_train_tfidf, y_train, cv=skf, scoring="f1_macro", n_jobs=-1)
print(f"CV Macro F1 : {cv_svc.mean():.4f} ± {cv_svc.std():.4f}")

# Train the LinearSVC model on the full training dataset
# The model learns how to separate the classes using the TF-IDF features
svc.fit(X_train_tfidf, y_train)

# Predict the class labels for the test set using the trained model
# Output: one predicted label for each test example
# LinearSVC has no predict_proba — use decision_function scores for roc_auc (OvR)
y_pred_svc  = svc.predict(X_test_tfidf)

# LinearSVC has no predict_proba — use decision_function scores for roc_auc (OvR)
y_proba_svc = svc.decision_function(X_test_tfidf)

# Compute evaluation metrics using the shared helper function
# y_test      -> true labels for the test set
# y_pred_svc  -> predicted class labels
# y_proba_svc -> decision scores used by the helper for ROC-AUC and related metrics
# The returned dictionary usually contains accuracy, precision, recall, F1, and ROC-AUC
svc_metrics = get_metrics(y_test, y_pred_svc, y_proba_svc)
results["LinearSVC"] = {"CV F1": cv_svc.mean(), "CV Std": cv_svc.std(),
                        "Time(s)": round(time.time() - t0, 1), **svc_metrics}
print(pd.Series(svc_metrics).round(4).to_string())

## Step 5 — Evaluation & Results

**Checking for:** CV F1 ≈ Test F1 (within ~1%) — confirms good generalisation, no overfitting to training set. Sorted by ROC-AUC.

---

### Why Macro F1 Is the Right Primary Metric for this projest.

**The dataset is balanced.** All 27 intent classes contain between 950 and 1,000 samples (~3.7% each). This means accuracy and macro F1 move together — a model cannot inflate accuracy by over-predicting a majority class because no majority class exists. Macro F1 is therefore not chosen *instead of* accuracy — it produces the same ranking here but is the more principled choice.

**Macro F1 treats all 27 classes equally.** Macro averaging computes F1 separately for each of the 27 intents and then takes the unweighted mean. This means that a rare confusion between cancel_order and change_order hurts the metric just as much as a common error would — no class is hidden inside a large support count. For intent routing, every class matters equally: misrouting a refund query is just as harmful as misrouting an account query.

**Macro F1 captures both precision and recall.** Accuracy only counts correct predictions. F1 penalises a model that achieves high accuracy by confidently mispredicting a small number of classes — the harmonic mean of precision and recall forces the model to be both precise (low false positives) and complete (low false negatives) across all 27 intents simultaneously.

**ROC-AUC is a useful secondary metric but not the primary one.** ROC-AUC (macro, one-vs-rest) measures ranking quality — how well the model separates each class from the other 26. It is useful for comparing probability calibration and for threshold analysis, but it does not directly reflect the classification decision the system makes at deployment. At prediction time, the model picks `argmax(softmax)` — that is a hard decision, which is what F1 evaluates. ROC-AUC also saturates near 1.0 for all models here, making it uninformative for ranking.

**Weighted F1 would be inappropriate here.** Weighted F1 scales each class's F1 by its support count. On a balanced dataset it produces nearly identical results to macro F1, but on a real-world imbalanced deployment corpus it would systematically underweight rare intents. Macro F1 is chosen to be robust to that future condition.

The **Final Model Comparison ML** step compares both classical machine learning models by evaluating;
*   Accuracy
*   Precision (macro)
*   Recall (macro)
*   F1- score (macro)
*   ROC-AUC (macro)
Results are displayed ina table for comaprison and exported in a CSV file  for reporting
The comparison provides a clear basis for choosing the best model before going onto Deep Learning







In [ ]:
# =============================================================================
# FINAL MODEL COMPARISON
# =============================================================================
section("Classical ML — final comparison")

res_df = pd.DataFrame(results).T
# Column order matches df_raw: accuracy | precision | recall | f1 | roc_auc
res_df = res_df[["accuracy", "precision", "recall", "f1", "roc_auc", "CV F1", "CV Std", "Time(s)"]]
res_df = res_df.sort_values("roc_auc", ascending=False)

print("=== FINAL MODEL COMPARISON ===")
# Select the main evaluation metrics for display
# .round(6) rounds the metric values to 6 decimal places since the results are often very close to each other and we want to see the differences clearly
# .to_string() prints the full DataFrame neatly as plain text
print(res_df[["accuracy", "precision", "recall", "f1", "roc_auc"]].round(6).to_string())
res_df.to_csv("/content/nb1_classical_ml_results.csv")

# summary table
styled_res = (
    res_df[["accuracy", "precision", "recall", "f1", "roc_auc"]].style
    .background_gradient(subset=["roc_auc"],   cmap="Greens")
    .background_gradient(subset=["f1"],        cmap="Blues")
    .background_gradient(subset=["precision"], cmap="Oranges")
    .set_properties(**{"background-color": "white", "color": "black", "border-color": "black"})
    .format("{:.6f}")
)
styled_res

The **TuningTraditional ML Models** step tunes the hyperparameters of the traditional Machine learning models (logistics regressions and Linear SVC) using GridSearchCV.
GridSearchCV trains the model and test different values of regularisation strength (0.1,1,10, etc). a small value like 0.1 avoids over fitting. And a high value can better train the data(using more samples) but risks overfitting

In [ ]:
# =============================================================================
# Hyperparameter tuning Traditional ML models — Logistic Regression, LinearSVC
# =============================================================================

section ("Hyperparameter tuning")

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

# defining the models and their respective hyperparameters to be tuned using GridSearchCV,
# which will perform an exhaustive search over the specified parameter grid for each model
# to find the best combination of hyperparameters based on cross-validation performance
trsditional_models = {
    "Logistic Regression":{
        "model": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "params": {
            "C": [0.1, 1, 3, 100],
            "penalty": ["l2"]
        }
    },
    "LinearSVC": {
        "model": LinearSVC(max_iter=3000, random_state=RANDOM_STATE),
        "params": {
            "C": [0.1, 1, 3, 100]
        }
    }
}

tuned_results={}
best_models={}

# Loop through each model and its hyperparameter setup, perform grid search to find the best hyperparameters,
# evaluate the best model on the test set, and store the results in a dictionary for comparison.

for model_name, setup in trsditional_models.items():
    print("\n" + "=" * 95)
    print(f"Tuning {model_name}")

    start = time.time()

    # grid search object that will perform an exhaustive search over the specified hyperparameter grid for the given model,
    # using cross-validation to evaluate each combination of hyperparameters based on the macro F1-score, and utilizing all available CPU cores for parallel processing
    grid= GridSearchCV(
        estimator=setup["model"],
        param_grid=setup["params"],
        scoring="f1_macro",
        cv=skf,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train_tfidf, y_train)

    best_model = grid.best_estimator_
    best_models[model_name] = best_model

    y_pred = best_model.predict(X_test_tfidf)

    # assigning metrics to a dictionary for the current model, including the best hyperparameters found by grid search,
    # the best cross-validation macro F1 score, and the test set evaluation metrics (accuracy, precision, recall, F1),
    # as well as the time taken for the entire tuning and evaluation process
    metrics={
        "Best params": grid.best_params_,
        "CV F1": grid.best_score_,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_test, y_pred, average="macro"),
        "Time(s)": round(time.time() - start, 1)
    }

    tuned_results[model_name]=metrics

    print("Best parameters:", grid.best_params_)
    print("Best CV Macro F1:", round(grid.best_score_, 4))
    print( "Test Macro F1:", round (metrics["F1"], 4))

Logistic Regression:With C= 10, the model achived a macro F1 score of 0.994 and validations data and .9943 on test data.

Linear SVC, C=3 was identied as the optimal LinearSVC and it achieved a macro F1 score of 0.994 on validation data and a score of 0.9946 on test data.

**Results Summary** this section summarises the performaceof the tunes traditiona machine learning models.
metrics are;
*   Cross-validates F1 Score (CV F1)
*   Test Accuracy, Precicion, Recall, F1 Score
*   Tuning time and evaulation time

This helps identify the best perfroming model



In [ ]:
# =============================================================================
# Results summary table for the tuned traditional ML models
# =============================================================================

section("Results summary: Traditional ML tuning")

#convert tuned results to dataframe
tuned_results_df = pd.DataFrame(tuned_results).T

#select evaluation metrics
display_cols = [
    "CV F1",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Time(s)"
]
tuned_summary_df =tuned_results_df[display_cols].sort_values("F1", ascending=False)
print(tuned_summary_df.round(4).to_string())
tuned_summary_df

This step generates final test predictions for the tuned classical machine learning models.

In [ ]:
# =============================================================================
# COMPUTE TUNED CLASSICAL PREDICTIONS
# Mirrors how y_pred_lr / y_proba_lr were stored after the baseline models.
# best_models and tuned_results were produced by the GridSearchCV section above.
# =============================================================================
section("Tuned classical ML — predictions & metrics")

# LR (tuned) - adding to final summary collection
y_pred_lr_tuned   = best_models["Logistic Regression"].predict(X_test_tfidf)
y_proba_lr_tuned  = best_models["Logistic Regression"].predict_proba(X_test_tfidf)
lr_tuned_metrics  = get_metrics(
    y_test, y_pred_lr_tuned, y_proba_lr_tuned,
    model_name="LR (tuned)", tier="Classical ML",
    time_s=tuned_results["Logistic Regression"]["Time(s)"]
)

# LinearSVC (tuned) -adding to final summary collection
y_pred_svc_tuned  = best_models["LinearSVC"].predict(X_test_tfidf)
y_proba_svc_tuned = best_models["LinearSVC"].decision_function(X_test_tfidf)
svc_tuned_metrics = get_metrics(
    y_test, y_pred_svc_tuned, y_proba_svc_tuned,
    model_name="LinearSVC (tuned)", tier="Classical ML",
    time_s=tuned_results["LinearSVC"]["Time(s)"]
)

This step calculates the best performing traditional machine learning model based on test set F1 score.
the model with the highest F1 socre is chosen and it will be used for the final evaulation and comaprison against other approaches


In [ ]:
chosen_traditional_model_name = tuned_summary_df["F1"].idxmax()
chosen_traditional_model = best_models[chosen_traditional_model_name]

print("Chosen Traditional ML Model:", chosen_traditional_model_name)
print("Chosen Macro F1:", round(tuned_summary_df.loc[chosen_traditional_model_name, "F1"], 4))
print("Chosen Parameters:", tuned_results[chosen_traditional_model_name]["Best params"])

**Model selection:** Traditional ML

# Short summary on the above output

The final comparison shows that both LinearSVC and Logistic Regression achieved near-perfect performance on the customer intent classification task. LinearSVC slightly outperformed Logistic Regression, with an F1-score of 0.9950 and ROC-AUC of 1.0000. Overall, both models demonstrated excellent predictive ability, with LinearSVC emerging as the best classical machine learning model.

## LinearSVC evaluation
This code evaluates the LinearSVC model at the individual intent level.

A classification report is generated for the test set, and the precision, recall, F1-score, and support for each intent are stored in a DataFrame. The results are then visualized in a horizontal bar chart, and the 5 hardest and 5 easiest intents are printed based on F1-score. This helps identify which customer intents the model predicts well and which ones remain more difficult.

The next cell visualises the comparison between corss validation and test performance for classical machine leaning models
*   **C1 Macro F1** shows the avearage perfromance across training folds
*   **Test Macro F1** represenst the proformance of unseen data


In [ ]:
# =============================================================================
# COMPARISON CHART — CV vs TEST MACRO F1
# =============================================================================
section("Comparison chart")
# Convert the index of `res_df` into a normal column
# `res_df` currently has model names as its index
# `.reset_index()` moves the index into a column
# `.rename(columns={"index": "Model"})` renames that new column to "Model"
# This makes the model names easier to use for plotting on the x-axis
res_plot = res_df.reset_index().rename(columns={"index": "Model"})

fig = go.Figure()
fig.add_trace(go.Bar(
    name="CV Macro F1",
    x=res_plot["Model"], y=res_plot["CV F1"],
    marker_color="#2E86AB",
    error_y=dict(type="data", array=res_plot["CV Std"].tolist(), visible=True)
))
fig.add_trace(go.Bar(
    name="Test Macro F1",
    x=res_plot["Model"], y=res_plot["f1"],
    marker_color="#D1495B"
))
fig.update_layout(
    barmode="group",
    title="Classical ML — CV vs Test Macro F1",
    title_x=0.02,
    yaxis=dict(title="Macro F1", range=[0.5, 1.0]),
    legend_title_text="Split"
)
fig.show()

## Above chart explanation
The blue bars are the average CV Macro F1 scores.
The red bars are the test Macro F1 scores.

The comparison chart shows that both LinearSVC and Logistic Regression achieved near-perfect Macro F1 scores on both cross-validation and the test set. The very small gap between CV and test performance suggests that both models generalize well and do not show clear signs of overfitting. In addition, the small error bars indicate stable performance across folds. Overall, both classical machine learning models performed extremely strongly on the intent classification task, with only minimal differences between them.

### Evaluation of each intent individual class function

It does three main things:

creates a classification report for all intent classes
builds a DataFrame containing per-intent F1, precision, recall, and support
visualizes the F1 score for each intent and prints the 5 hardest and 5 easiest intents

This is useful because a model can have a strong overall score while still performing poorly on a few specific intent classes

This step evaluates the model peroformance at the individual intent level by using the precision, recall, and the F1-Score for each class. it shows the easiets and hardest intents to classify and it shows here the model performs well or where it struggles.

In [ ]:
# =============================================================================
# PER-INTENT F1 TABLE — LinearSVC
# =============================================================================
section("Per-intent F1 breakdown — LinearSVC")

import re as _re

# A classification report function to extract precision, recall, F1, and support for each class
# and it returns the report as a Python dictionary instead of plain text
report_dict = classification_report(y_test, y_pred_svc, target_names=le.classes_, output_dict=True)
intent_f1 = pd.DataFrame({
    "Intent":    le.classes_,
    "F1 Score":  [report_dict[c]["f1-score"] for c in le.classes_],
    "Precision": [report_dict[c]["precision"] for c in le.classes_],
    "Recall":    [report_dict[c]["recall"] for c in le.classes_],
    "Support":   [report_dict[c]["support"] for c in le.classes_],
}).sort_values("F1 Score")

fig = px.bar(
    intent_f1,
    x="F1 Score", y="Intent",
    orientation="h",
    title="Per-intent F1 score — LinearSVC",
    color="F1 Score",
    color_continuous_scale="RdYlGn",
    labels={"F1 Score": "F1 Score"}
)
fig.update_layout(title_x=0.02, height=700, yaxis_title=None)
fig.show()

print("\n5 hardest intents:")
print(intent_f1.head(5)[["Intent","F1 Score","Precision","Recall"]].round(4).to_string(index=False))
print("\n5 easiest intents:")
print(intent_f1.tail(5)[["Intent","F1 Score","Precision","Recall"]].round(4).to_string(index=False))

### Key conclusion
The per-intent analysis shows that LinearSVC is highly effective across all customer-support intents. Most classes achieve near-perfect or perfect F1 scores, while only a small number of semantically similar intents, such as change_order and cancel_order, are slightly more difficult to classify.

### Confusion Matrix — LinearSVC

**Checking for:** Which intent pairs are most confused.

**Expected pattern:** Confusion concentrates between semantically similar intents in the same category — cancel_order vs change_order, track_order vs track_refund. These share vocabulary that TF-IDF cannot distinguish. Cross-category errors are rare.

This step shows displays the model performance using confuison matrix. it show how well and intent is classififed and show patterns in miscalcualtions also.

In [ ]:
# =============================================================================
# CONFUSION MATRIX — BEST MODEL (LinearSVC)
# =============================================================================
section("Confusion matrix — LinearSVC (best classical model)")

print(classification_report(y_test, y_pred_svc, target_names=le.classes_))

cm_norm = confusion_matrix(y_test, y_pred_svc, normalize='true')

fig = px.imshow(
    cm_norm,
    title="Normalised confusion matrix — LinearSVC (test set)",
    color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="True", color="Proportion"),
    aspect="auto",
    height=750,
    zmin=0, zmax=1,
)
fig.update_layout(title_x=0.02)
fig.show()

### Explanation to the above diagram:

The dataset is a multi-class classification with many intent labels,the confusion matrix for LinearSVC shows a strong concentration of predictions along the main diagonal, indicating that most test examples were classified correctly. Only a small number of off-diagonal errors are visible, suggesting very limited confusion between intent classes. This confirms the strong overall performance observed in the classification report and macro F1 score.

This step shows the impact of reducing IT-IDF on model performance.

In [ ]:
# =============================================================================
# DATA REDUCTION ANALYSIS — TF-IDF feature reduction
# Sweeping max_features to test how vocabulary size affects classical ML
# =============================================================================
section("Data reduction analysis — TF-IDF feature sweep")

from sklearn.feature_extraction.text import TfidfVectorizer

vocab_sizes = [500, 1000, 2000, 5000, 10000, 50000]
reduction_rows = []

for max_feat in vocab_sizes:
    # Refit TF-IDF with reduced vocabulary
    tfidf_r = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=max_feat,
        sublinear_tf=True,
        min_df=2,
        strip_accents="unicode",
    )
    X_train_r = tfidf_r.fit_transform(X_train)
    X_test_r = tfidf_r.transform(X_test)
    actual_vocab = len(tfidf_r.vocabulary_)

    # LR
    lr_r = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                               solver="saga", n_jobs=-1)
    lr_r.fit(X_train_r, y_train)
    f1_lr = f1_score(y_test, lr_r.predict(X_test_r), average="macro")

    # LinearSVC
    svc_r = LinearSVC(max_iter=2000, random_state=RANDOM_STATE, C=1.0)
    svc_r.fit(X_train_r, y_train)
    f1_svc = f1_score(y_test, svc_r.predict(X_test_r), average="macro")

    reduction_rows.append({
        "max_features": max_feat,
        "actual_vocab": actual_vocab,
        "LR_F1": f1_lr,
        "LinearSVC_F1": f1_svc,
    })

reduction_df = pd.DataFrame(reduction_rows)
print(reduction_df.round(4).to_string(index=False))

# Plot — F1 vs vocabulary size
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=reduction_df["actual_vocab"], y=reduction_df["LR_F1"],
    mode="lines+markers", name="Logistic Regression",
    line=dict(color="#2E86AB", width=2),
))
fig.add_trace(go.Scatter(
    x=reduction_df["actual_vocab"], y=reduction_df["LinearSVC_F1"],
    mode="lines+markers", name="LinearSVC",
    line=dict(color="#D1495B", width=2),
))
fig.update_layout(
    title="Data reduction analysis — Macro F1 vs TF-IDF vocabulary size",
    title_x=0.02,
    xaxis_title="Actual vocabulary size",
    yaxis_title="Test Macro F1",
    xaxis_type="log",
)
fig.show()

**Reading the data-reduction analysis.** The TF-IDF vocabulary sweep shows that performance plateaus at around 2,000 features — both
models gain less than 0.001 F1 beyond that point. This is because the dataset only
produces 4,901 distinct patterns after filtering, so the 50,000-feature cap was never
reached. Even at 500 features both models remain above F1 = 0.97, confirming the
customer-support vocabulary is narrow and highly discriminative. A 2,000-feature TF-IDF
model would lose less than 0.001 F1 compared to the default configuration while using
less than half the features.

# Model Developmet: BiLSTM model

Below step is to define a reusable function that creates the BiLSTM classifier. The model uses an embedding layer, a bidirectional LSTM encoder, a pooling layer, and a dense classification head.

This structure is suitable for customer-support text because the meaning of a sentence often depends on both earlier and later words in the sequence.

## Input summary

The BiLSTM receives padded token sequences as input and predicts one of the encoded intent classes.
The variables below were prepared earlier in the notebook and are reused here:

1. X_train_seq, X_val_seq, X_test_seq: padded token sequences
2. y_train, y_val, y_test: encoded target labels
3. MAX_LEN: maximum sequence length
4. EMBED_DIM: embedding size
5. NUM_CLASSES: number of intent classes

## Define the BiLSTM model

The next cell defines a reusable function that builds the BiLSTM classifier. The model includes:

1. an embedding layer
2. spatial dropout for regularisation
3. a bidirectional LSTM layer
4. global max pooling
5. a dense classification head
6. a softmax output layer

This structure is appropriate for short customer-support messages because the important intent cues may appear anywhere in the sequence.

## BiLSTM setup

A small helper function is also defined to check whether the GloVe embedding matrix is available. This is useful because the model should still work even if pretrained embeddings are missing.

Pretrained GloVe was chosen for three reasons.

**First**, the Bitext training set contains only ~18,800 utterances after splitting — far too few to learn useful embeddings from scratch for a 2,368-word vocabulary. Random initialisation would force the model to spend most of its capacity learning that *cancel* and *refund* are different words, rather than learning the intent-classification task itself.

**Second**, GloVe vectors already encode general-purpose lexical semantics learned from billions of co-occurrence statistics, so the model starts with a strong prior — synonyms cluster together, and morphological variants (*cancel / cancelled / cancelling*) point in similar directions. This gives faster convergence and better generalisation to unseen phrasings in the test set.

**Third**, the 100-dimensional variant was selected as a deliberate balance: large enough to carry meaningful semantic structure, small enough to keep the embedding matrix compact (~24k × 100 = 2.4M parameters) and avoid overfitting on a small training corpus.

---

The embedding matrix is built by looking up each tokeniser-vocabulary word in the GloVe dictionary; words not found (out-of-vocabulary) are left as zero vectors. The OOV rate is reported in the notebook as a sanity check, a high OOV rate would indicate that GloVe is a poor match for the customer-support domain and that domain-specific embeddings (e.g. fastText trained on support tickets) should be considered instead.

By default the embedding layer is **frozen** during training so the pretrained semantics are preserved; one tuning trial **unfreezes** the layer to allow domain adaptation, and the trade-off between the two is reported in the BiLSTM tuning table.

This step deines the Bidirectional LSTM  using pretrained GloVe embeddings. The model processes tokenised sequences.

In [ ]:
# =============================================================================
# BILSTM MODEL BUILDER
# =============================================================================

section("Define BiLSTM model")

glove_available = GLOVE_PATH
# Sanity check — fail fast if GloVe didn't load, rather than silently using zeros
assert embedding_matrix is not None and np.any(embedding_matrix != 0), (
    "GloVe embedding matrix is empty — check GLOVE_PATH in the embedding cell."
)

# function to build a BiLSTM text classification model using Keras, with configurable hyperparameters for the LSTM and dense layers,
# dropout rates, learning rate, and whether to use trainable embeddings. The model architecture includes an embedding layer initialized with GloVe vectors,
# a spatial dropout layer for regularization, a bidirectional LSTM layer to capture sequential information,
# a global max pooling layer to extract the strongest features, and a dense classification head that outputs probabilities
# for each class using softmax activation. The model is compiled with the Adam optimizer and sparse categorical cross-entropy loss.
def build_bilstm_model(
    lstm_units=128,
    dense_units=64,
    lstm_dropout=0.2,
    dense_dropout=0.4,
    spatial_dropout=0.2,
    learning_rate=1e-3,
    trainable_embeddings=False, # kept as false and later changed it to trainable_embedding to test both true and false scenario
    model_name="BiLSTM",
):
    """
    Build a BiLSTM text classifier using the existing notebook variables.
    """

    # glove file assignment
    glove_available = GLOVE_PATH

    model = models.Sequential(name="BiLSTM_classifier")

    # Input = padded sequence of token ids
    model.add(layers.Input(shape=(MAX_LEN,), name="input_ids"))

    # Embedding layer
    model.add(
        layers.Embedding(
        input_dim=vocab_size,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        trainable=trainable_embeddings,
        mask_zero=True,
        name="embedding"
        )
    )

    # Regularisation at the embedding level
    model.add(layers.SpatialDropout1D(spatial_dropout, name="spatial_dropout"))

    # Bidirectional LSTM
    model.add(
        layers.Bidirectional(
            layers.LSTM(
                lstm_units,
                dropout=lstm_dropout,
                return_sequences=True
            ),
            name="bilstm"
        )
    )

    # Pool strongest signal across the sequence
    model.add(layers.GlobalMaxPooling1D(name="global_max_pool"))

    # Dense classification head
    model.add(layers.Dense(dense_units, activation="relu", name="dense_relu"))
    model.add(layers.Dropout(dense_dropout, name="dense_dropout"))

    # Final multi-class classifier
    model.add(layers.Dense(NUM_CLASSES, activation="softmax", name="classifier"))

    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

print("build_bilstm_model() ready ✓")

### Why this BiLSTM architecture was chosen

This architecture was selected because it is *appropriate for multi-class text classification.*

The embedding layer converts token IDs into dense vector representations. If GloVe vectors are available, the model starts from pretrained semantic information. The parameter trainable_embeddings allows the experiment to compare frozen embeddings against fine-tuned embeddings.

**SpatialDropout1D** is used to improve generalisation by dropping parts of embedding features during training.

The Bidirectional LSTM reads the message in both directions. This is useful because important intent cues may appear at the beginning, middle, or end of a customer message.

**GlobalMaxPooling1D** reduces the sequence output into one fixed-size vector by keeping the strongest feature activations. This often works well for intent classification because a few important words or phrases can be highly informative.

The output layer uses softmax because this is a multi-class classification problem with NUM_CLASSES intent labels.

The loss function is sparse categorical cross-entropy because the target labels are already integer encoded rather than one-hot encoded.

### Training helper for BiLSTM experiments

The following function trains one BiLSTM configuration and returns:

1. The trained model
2. The training history
3. The training history
4. A validation summary used for hyperparameter tuning
the total training time

The validation metrics are computed directly inside the function so that the shared helper get_metrics() can remain the common final evaluation function for all models.

This step defines the resuable functio for testing different BiLSTM confiugurations. It evaluates performance validation performance and returns training history, mtrics, runtime, etc

In [ ]:
# =============================================================================
# TRAINING HELPER FOR BILSTM
# =============================================================================
section("BiLSTM training helper")

def train_one_bilstm(config, verbose=0):
    """
    Training one BiLSTM configuration and evaluating on the validation set.
    Returns model, history, validation summary, and training time.
    """
    # K is the Keras backend module, and gc is the garbage collection module.
    # gc is used to free up memory by collecting and disposing of unused objects,
    # which is important when training multiple models in a loop to prevent memory leaks and ensure that each model starts with a clean slate.
    K.clear_session()
    gc.collect()

    # setting up model hyperparameters based on the provided configuration dictionary,
    # which includes parameters for the LSTM and dense layers, dropout rates, learning rate, and whether to use trainable embeddings.
    # The model is built using the previously defined `build_bilstm_model` function with these hyperparameters.
    model = build_bilstm_model(
        lstm_units=config["lstm_units"],
        dense_units=config["dense_units"],
        lstm_dropout=config["lstm_dropout"],
        dense_dropout=config["dense_dropout"],
        spatial_dropout=config["spatial_dropout"],
        learning_rate=config["learning_rate"],
        trainable_embeddings=config["trainable_embeddings"]
    )

    cbs = [
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-5,
            verbose=1
        )
    ]

    start = time.time()

    history = model.fit(
        X_train_seq, y_train,
        validation_data=(X_val_seq, y_val),
        epochs=15,
        batch_size=config["batch_size"],
        callbacks=cbs,
        verbose=verbose
    )

    train_time = time.time() - start

    # Validation predictions for tuning
    val_proba = model.predict(X_val_seq, verbose=0)
    val_pred = val_proba.argmax(axis=1)

    val_summary = {
        "trial_name": config["trial_name"],
        "lstm_units": config["lstm_units"],
        "dense_units": config["dense_units"],
        "lstm_dropout": config["lstm_dropout"],
        "dense_dropout": config["dense_dropout"],
        "spatial_dropout": config["spatial_dropout"],
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "trainable_embeddings": config["trainable_embeddings"] if glove_available else True,
        "epochs_ran": len(history.history["loss"]),
        "best_val_loss": float(np.min(history.history["val_loss"])),
        "train_time_s": round(train_time, 1),
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_precision": precision_score(y_val, val_pred, average="macro", zero_division=0),
        "val_recall": recall_score(y_val, val_pred, average="macro", zero_division=0),
        "val_f1": f1_score(y_val, val_pred, average="macro"),
        "val_roc_auc": roc_auc_score(y_val, val_proba, multi_class="ovr", average="macro")
    }

    return model, history, val_summary, train_time

print("train_one_bilstm() ready ✓")

### Why this training function is needed

This helper function makes the notebook easier to manage because it standardises how each BiLSTM configuration is trained and compared.

***EarlyStopping*** is used to stop training when validation loss stops improving, which helps reduce overfitting.

***ReduceLROnPlateau*** lowers the learning rate when progress slows down, which can improve optimisation.

The model is tuned using validation macro F1-score because this is a multi-class classification task, and macro F1 gives balanced importance to each intent class..

## Define the hyperparameter search space

The next cell defines a small but meaningful tuning space for the BiLSTM. The selected hyperparameters control the model capacity, regularisation strength, optimisation behaviour, and whether the pretrained embeddings are frozen or fine-tuned.

###Why these hyperparameters were selected

A hyperparameter search is used here to keep the tuning process efficient while still exploring the most important BiLSTM design choices:

1. smaller vs larger BiLSTM capacity
2. frozen vs trainable embeddings
3. moderate differences in dropout and learning rate

This is sufficient for the assignment and keeps the experiments interpretable.

The four trials were chosen to test two main questions: whether a larger BiLSTM helps, and whether fine-tuning pretrained embeddings helps. The settings were kept mostly consistent so that the effect of each change could be interpreted clearly.

This step defines the BiLSTM configurations tested during hyperparemeter tuning.
the step compare froxen vs fine-tuned GloVe embeddings diffenrent LSTM sizes, droput settings, and learning rates

In [ ]:
# =============================================================================
# HYPERPARAMETER SEARCH SPACE
# =============================================================================
section("BiLSTM hyperparameter search space")

bilstm_search_space = [
{
"trial_name": "baseline_frozen",# Baseline setting: start with a small, stable BiLSTM and keep GloVe frozen to measure the default pretrained-embedding performance
"lstm_units": 64, # 64 units give enough sequence capacity for short support messages without making the model too heavy
"dense_units": 64, # A 64-unit dense layer is a simple classification head and matches the scale of the recurrent layer
"lstm_dropout": 0.2,# Mild dropout is used to reduce overfitting while still allowing the model to learn normally
"dense_dropout": 0.3,# Slightly stronger dropout is used in the dense layer because the classifier head can overfit quickly
"spatial_dropout": 0.2, # Spatial dropout is added to regularise embedding features before they enter the BiLSTM
"learning_rate": 1e-3, # Standard Adam learning rate, suitable when embeddings are frozen and fewer parameters are being updated
"batch_size": 64, # Chosen as a practical batch size that trains efficiently and remains stable
"trainable_embeddings": False # Embeddings are frozen here so the first trial reflects pure use of pretrained GloVe
},

{
"trial_name": "frozen_larger", # This trial changes only the BiLSTM size to check whether extra recurrent capacity helps when embeddings remain frozen
"lstm_units": 128, # Increased to 128 to test a larger encoder against the 64-unit baseline
"dense_units": 64, # Kept unchanged so the comparison focuses on the effect of the larger BiLSTM
"lstm_dropout": 0.2, # Same dropout as baseline to keep the trial comparable
"dense_dropout": 0.3, # Same dense dropout as baseline for a fair size-only comparison
"spatial_dropout": 0.2, # Same embedding regularisation as baseline
"learning_rate": 1e-3, # Kept the same because embeddings are still frozen and training conditions are otherwise similar
"batch_size": 64, # Kept the same to avoid introducing another change into the comparison
"trainable_embeddings": False # Still frozen, because this trial is meant to isolate the effect of model size
},

{
"trial_name": "finetune_small",# This trial keeps the baseline size but unfreezes GloVe to test whether domain-specific fine-tuning improves performance
"lstm_units": 64, # Kept at 64 so any gain can be linked mainly to embedding fine-tuning rather than model size
"dense_units": 64, # Kept unchanged for the same reason
"lstm_dropout": 0.2, # Same recurrent dropout as the baseline
"dense_dropout": 0.3, # Same dense dropout as the baseline
"spatial_dropout": 0.2,# Same embedding regularisation as the baseline
"learning_rate": 5e-4, # Reduced because pretrained embeddings usually need smaller updates when they are unfrozen
"batch_size": 64, # Left unchanged so the trial differs mainly in embedding behaviour
"trainable_embeddings": True # Unfreezing allows GloVe vectors to adapt to customer-support language
},

{
"trial_name": "finetune_medium", # Final trial combines embedding fine-tuning with a larger BiLSTM to test a slightly stronger model
"lstm_units": 128,# Increased to 128 to see whether a larger encoder helps once embeddings are also trainable
"dense_units": 64,# Kept fixed so the main extra capacity comes from the recurrent layer
"lstm_dropout": 0.2,# Same recurrent dropout as earlier trials
"dense_dropout": 0.4, # Increased slightly because a larger fine-tuned model has higher overfitting risk
"spatial_dropout": 0.2, # Same spatial dropout as the other trials
"learning_rate": 5e-4,# Same reduced learning rate used for fine-tuning pretrained embeddings
"batch_size": 64,# Kept fixed for consistency across all four trials
"trainable_embeddings": True # Embeddings are trainable here to test full fine-tuning with the larger BiLSTM
},
]

pd.DataFrame(bilstm_search_space)

### Why these four trials were chosen

The first trial, baseline_frozen, is the reference model.
The second, frozen_larger, tests whether increasing the number of LSTM units improves performance while keeping embeddings fixed.
The third, finetune_small, tests whether allowing pretrained embeddings to update during training improves the baseline.
The fourth, finetune_medium, combines a larger recurrent layer with trainable embeddings and slightly stronger dropout.

Together, these four trials provide a clear and manageable tuning study.

## Run hyperparameter tuning

The following cell trains each BiLSTM configuration and compares them using validation performance. The best configuration is selected using validation macro F1-score, while validation loss is used as a tie-breaker.

The test set is not used during this stage.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers, callbacks, models
import tensorflow.keras.backend as K
import gc

This code runs several BiLSTM hyperparameter trials, evaluates each one on the validation set, and keeps track of the best-performing configuration using validation macro F1. It then stores all trial results in a comparison table and prints the final best BiLSTM setup.

In [ ]:
# =============================================================================
# RUN HYPERPARAMETER TUNING
# =============================================================================
section("Run BiLSTM hyperparameter tuning")

# Create an empty list to store the validation summary results
# from each BiLSTM hyperparameter trial
tuning_rows = []
best_config = None
# Initialize the best validation F1 score with a very low value
# This ensures that the first real model score will replace it
best_val_f1 = -1
best_val_loss = np.inf

# Loop through each hyperparameter configuration in the search space
# `i` gives the trial number starting from 1
# `config` contains the hyperparameters for the current trial
for i, config in enumerate(bilstm_search_space, start=1):
    print("\n" + "-" * 95)
    print(f"Trial {i}/{len(bilstm_search_space)} : {config['trial_name']}")
    print(config)

    # Train one BiLSTM model using the current hyperparameter configuration
    # The function returns multiple outputs
    # Only the `summary` dictionary is kept here
    # `_` is used to ignore the other returned values
    _, _, summary, _ = train_one_bilstm(config=config, verbose=0)
    tuning_rows.append(summary)

    print(f"Val accuracy : {summary['val_accuracy']:.4f}")
    print(f"Val F1       : {summary['val_f1']:.4f}")
    print(f"Val ROC-AUC  : {summary['val_roc_auc']:.4f}")
    print(f"Best val loss: {summary['best_val_loss']:.4f}")
    print(f"Epochs ran   : {summary['epochs_ran']}")

    # Check whether the current trial is better than the best one seen so far
    # Primary criterion: higher validation F1
    # Tie-breaker: lower validation loss
    if (summary["val_f1"] > best_val_f1) or (
        np.isclose(summary["val_f1"], best_val_f1) and summary["best_val_loss"] < best_val_loss
    ):
        best_val_f1 = summary["val_f1"]
        best_val_loss = summary["best_val_loss"]
        best_config = config.copy()
# Convert the list of trial summaries into a pandas DataFrame
bilstm_tuning_df = (
    pd.DataFrame(tuning_rows)
    .sort_values(["val_f1", "val_accuracy", "val_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

print("\nBest BiLSTM configuration:")
print(best_config)

print("\nValidation comparison table:")
print(
    bilstm_tuning_df[
        [
            "trial_name", "lstm_units", "dense_units", "trainable_embeddings",
            "batch_size", "learning_rate", "epochs_ran",
            "val_accuracy", "val_f1", "val_roc_auc", "best_val_loss"
        ]
    ].round(4).to_string(index=False)
)

### Explanation of the above result

Four BiLSTM configurations were evaluated on the validation set. The tuning results show that models with trainable embeddings outperformed models with frozen embeddings. The best-performing configuration was finetune_small, which achieved a validation accuracy and macro F1-score of 0.9953 with the lowest validation loss of 0.0162. This suggests that helping the model understand the words better was more useful than just making the sequence model bigger.

Note: The ReduceLROnPlateau callback automatically lowered the learning rate when validation loss stopped improving. This helps the optimiser make smaller updates in later epochs and can improve final convergence.

## Hyperparameter tuning results

Tuning results

This step identifies the best-performing BiLSTM configuration using validation performance only. This approach avoids test-set leakage and ensures that the final evaluation remains fair.

The main selection metric is macro F1-score, because it reflects balanced performance across all intent classes.

## Visual comparison of validation performance

A visual comparison makes it easier to show how each BiLSTM configuration performed during tuning. The chart compares all F1 validation scores acress the different BiLSTM configurations

In [ ]:
# =============================================================================
# VISUALISE TUNING RESULTS
# =============================================================================
section("BiLSTM tuning comparison")

fig = px.bar(
    bilstm_tuning_df.sort_values("val_f1", ascending=True),
    x="val_f1",
    y="trial_name",
    orientation="h",
    color="trainable_embeddings",
    title="BiLSTM validation macro F1 by trial",
    labels={"val_f1": "Validation macro F1", "trial_name": "Trial"}
)
fig.update_layout(title_x=0.02, yaxis_title=None)
fig.show()

## Above result explanation
The tuning comparison chart shows the validation macro F1 scores for all BiLSTM configurations tested. The two models with trainable embeddings achieved higher validation performance than the models with frozen embeddings. The best-performing configuration was finetune_small, indicating that fine-tuning pretrained embeddings was more beneficial than simply increasing the size of the LSTM layer.

finetune_smal and finetune_medium is True and baseline_frozen and frozen_larger is false, the visual pattern sugges fine-tuning the embeddings improved validation performance

## Compare the baseline and tuned BiLSTM

After tuning, the baseline BiLSTM and the best tuned BiLSTM are trained and compared directly. This helps show whether hyperparameter tuning resulted in a meaningful performance improvement.

In [ ]:
# =============================================================================
# FINAL TRAINING — BASELINE VS TUNED
# =============================================================================
section("Final BiLSTM training: baseline vs tuned")

# Select the first configuration from the BiLSTM hyperparameter search space
# This is being used as the baseline model configuration
# In your tuning list, this is usually the simpler reference setup
# such as `baseline_frozen
baseline_config = bilstm_search_space[0]

# Train the baseline BiLSTM model using the selected baseline configuration
# train_one_bilstm(...) returns four outputs:
# - baseline_model       -> the trained baseline BiLSTM model
# - baseline_history     -> training history object containing loss/metric values across epochs
# - baseline_val_summary -> summary of validation performance for the baseline model
# - baseline_time        -> total training time for the baseline model
# config=baseline_config -> pass the baseline hyperparameter settings into the training function
# verbose=0              -> suppress per-epoch training logs for cleaner notebook output
baseline_model, baseline_history, baseline_val_summary, baseline_time = train_one_bilstm(
    config=baseline_config,
    verbose=0
)

# Train the tuned BiLSTM model using the best hyperparameter configuration
# `best_config` was selected earlier from the tuning results
# This call also returns four outputs:
# - tuned_model       -> the trained best/tuned BiLSTM model
# - tuned_history     -> training history for the tuned model
# - tuned_val_summary -> validation summary for the tuned model
# - tuned_time        -> total training time for the tuned model
# This allows direct comparison between the baseline model and the tuned model
tuned_model, tuned_history, tuned_val_summary, tuned_time = train_one_bilstm(
    config=best_config,
    verbose=0
)

plot_history(baseline_history, "BiLSTM (baseline)")
plot_history(tuned_history, "BiLSTM (tuned)")

## Above output explanation

The loss curves show that both the baseline and tuned BiLSTM models learned effectively, with training and validation loss decreasing rapidly in the early epochs and stabilizing at low values. The tuned model achieved slightly lower final loss while maintaining stable validation performance, indicating improved convergence without clear signs of overfitting.

## Why the baseline comparison is important

The baseline model acts as a reference point. Without a baseline, it would be difficult to show whether the tuning process truly improved the BiLSTM.

The training curves also help assess whether the models converged properly and whether overfitting occurred.

## Final test evaluation using the shared evaluation helper

The held-out test set is used only once at the end of the tuning process. The existing shared get_metrics() helper is reused so that evaluation remains consistent across all models in the project.

In [ ]:
# =============================================================================
# TEST SET EVALUATION — USE SHARED HELPER
# =============================================================================
section("BiLSTM test evaluation")

# Use the trained baseline BiLSTM model to predict class probabilities
# for the test set
# X_test_seq -> padded token sequences for the test data
# verbose=0  -> suppress prediction progress output
# Output: for each test sample, a probability is returned for every class
baseline_proba = baseline_model.predict(X_test_seq, verbose=0)

# Convert the baseline model's probability outputs into final predicted class labels
# argmax(axis=1) selects the class with the highest predicted probability
# for each test example
baseline_pred = baseline_proba.argmax(axis=1)

# Use the tuned BiLSTM model to predict class probabilities for the test set
# Output: probability distribution over all intent classes for each test sample
tuned_proba = tuned_model.predict(X_test_seq, verbose=0)

# Convert the tuned model's class probabilities into predicted class labels
# by choosing the class with the highest probability for each sample
tuned_pred = tuned_proba.argmax(axis=1)

# Evaluate the baseline BiLSTM model using the shared get_metrics() helper
# y_true=y_test              -> true labels for the test set
# y_pred=baseline_pred       -> predicted class labels from the baseline model
# y_proba=baseline_proba     -> predicted class probabilities from the baseline model
# model_name="BiLSTM (baseline)" -> label used to identify the model in the output
# tier="Deep Learning"       -> category of model, useful for result grouping
# time_s=baseline_time       -> total training time for the baseline model
# The returned dictionary usually includes accuracy, precision, recall, F1, ROC-AUC, and time
baseline_metrics = get_metrics(
    y_true=y_test,
    y_pred=baseline_pred,
    y_proba=baseline_proba,
    model_name="BiLSTM (baseline)",
    tier="Deep Learning",
    time_s=baseline_time
)

# same evaluation done for tuned
tuned_metrics = get_metrics(
    y_true=y_test,
    y_pred=tuned_pred,
    y_proba=tuned_proba,
    model_name="BiLSTM (tuned)",
    tier="Deep Learning",
    time_s=tuned_time
)

# Store the baseline BiLSTM results in the shared `results` dictionary
# Key: "BiLSTM (baseline)"
# Value: dictionary of evaluation metrics
results["BiLSTM (baseline)"] = baseline_metrics

# Store the tuned BiLSTM results in the shared `results` dictionary
# Key: "BiLSTM (tuned)"
# Value: dictionary of evaluation metrics
results["BiLSTM (tuned)"] = tuned_metrics

# Create a DataFrame containing one row for the baseline model and one row for the tuned model
bilstm_test_compare = (
    pd.DataFrame([baseline_metrics, tuned_metrics])
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

print("\nTest comparison:")
print(
    bilstm_test_compare[
        ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "time_s"]
    ].round(4).to_string(index=False)
)

## Output Explanation

The final test evaluation shows that both BiLSTM models performed extremely well on the intent classification task. However, the tuned BiLSTM slightly outperformed the baseline across all evaluation metrics, achieving an accuracy and macro F1-score of 0.9955 compared with 0.9926 for the baseline. This indicates that hyperparameter tuning improved the model’s ability to classify customer messages into the correct support intent while maintaining strong generalization performance

## Final performance comparison

The final test comparison reports the main classification metrics:

1. accuracy
2. macro precision
3. macro recall
4. macro F1-score
5. ROC-AUC
6. training time

Among these, macro F1-score is the most important metric because it gives balanced importance to each intent class.

## Classification report for the tuned BiLSTM

A detailed classification report is generated for the tuned BiLSTM. This helps identify which intent classes are predicted well and which classes remain more difficult.

In [ ]:
# =============================================================================
# CLASSIFICATION REPORT — TUNED MODEL
# =============================================================================
section("Classification report — BiLSTM (tuned)")

print(
    classification_report(
        y_test,
        tuned_pred,
        target_names=le.classes_,
        digits=4,
        zero_division=0
    )
)

## Confusion matrix for the tuned BiLSTM

The confusion matrix below provides a visual summary of prediction errors and helps identify which intent classes are commonly confused with one another.

In [ ]:
# =============================================================================
# CONFUSION MATRIX — TUNED MODEL
# =============================================================================
section("Confusion matrix — BiLSTM (tuned)")
# A normalization of the matrix is done so that the output can be easily interupt
cm = confusion_matrix(y_test, tuned_pred, normalize='true')

fig = px.imshow(
    cm,
    color_continuous_scale="Blues",
    aspect="auto",
    labels=dict(x="Predicted label", y="True label", color="Count"),
    title="BiLSTM (tuned) — confusion matrix"
)
fig.update_layout(title_x=0.02)
fig.show()

Th step below evaluates the performance of the best classical model using a classfiation report and a confuison matrix

In [ ]:
# =============================================================================
# CONFUSION MATRIX — BEST MODEL (LinearSVC)
# =============================================================================
section("Confusion matrix — LinearSVC (best classical model)")

print(classification_report(y_test, y_pred_svc, target_names=le.classes_))

cm_norm = confusion_matrix(y_test, y_pred_svc, normalize='true')
cm_norm_df = pd.DataFrame(cm_norm, index=le.classes_, columns=le.classes_)

fig = px.imshow(
    cm_norm_df,
    title="Normalised confusion matrix — LinearSVC (test set)",
    color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="True", color="Proportion"),
    aspect="auto",
    height=750,
    zmin=0, zmax=1,
)
fig.update_traces(
    hovertemplate="True: %{y}<br>Predicted: %{x}<br>Proportion: %{z:.4f}<extra></extra>"
)
fig.update_layout(title_x=0.02)
fig.show()

# **Model Development: Gated Recurrent Units (GRU)**

The next step is to define a reusable function that creates the GRU classifier. The model uses an embedding layer, a bidirectional GRU encoder, a pooling layer, and a dense classification head.

This structure is suitable for customer-support text because the meaning of a sentence often depends on both earlier and later words in the sequence, while the GRU offers a simpler recurrent architecture than LSTM with fewer parameters.

**Input summary**

The GRU receives padded token sequences as input and predicts one of the encoded intent classes. The variables below were prepared earlier in the notebook and are reused here:

X_train_seq, X_val_seq, X_test_seq: padded token sequences
y_train, y_val, y_test: encoded target labels
MAX_LEN: maximum sequence length
EMBED_DIM: embedding size
NUM_CLASSES: number of intent classes

**Why GRU was also included**

The GRU was developed alongside the BiLSTM to provide a meaningful comparison between two standard recurrent deep learning architectures for text classification. While both models are designed to capture sequential information, the GRU has a simpler gating mechanism and fewer parameters than the LSTM. This can make it faster to train and less prone to overfitting, especially on smaller datasets.

For this project, the GRU is useful as a second recurrent baseline because it allows comparison of whether the additional complexity of the LSTM provides any practical benefit over a lighter recurrent architecture for customer-intent routing.

The GRU model was implemented as a second standard deep learning architecture for intent classification. It uses the same padded token-sequence inputs and GloVe-initialised embedding matrix as the BiLSTM, but replaces the LSTM encoder with a bidirectional GRU layer. This provides a lighter recurrent architecture for comparison while still capturing contextual information from both directions of the customer message

This step defines the GRU text classification model using pretrained GloVE embedding. The GRU is a lighter recurring alternative to BiLSTM- this provides a comparison, taining efficiency and model complexity.

In [ ]:
section("GRU Model Definition")

glove_available = GLOVE_PATH
# Sanity check — fail fast if GloVe didn't load, rather than silently using zeros
assert embedding_matrix is not None and np.any(embedding_matrix != 0), (
    "GloVe embedding matrix is empty — check GLOVE_PATH in the embedding cell."
)
'''
GRU is a lighter recurrent unit than LSTM — it has 2 gates instead of 3,
    so it trains faster with similar accuracy on short text. Used here as a
    direct comparison point against BiLSTM to show whether bidirectionality
    + extra gating is worth the extra parameters on this dataset.
'''

def build_gru_model(
    gru_units=128,
    dense_units=64,
    gru_dropout=0.2,
    dense_dropout=0.4,
    spatial_dropout=0.2,
    learning_rate=1e-3,
    trainable_embeddings=False,
    model_name="GRU",
):

#Build a GRU text classifier using the existing notebook variables.


  model_gru = models.Sequential(name="GRU_classifier")

#input layer
  model_gru.add(layers.Input(shape=(MAX_LEN,), name="input_ids"))
  #Embedding layer
  model_gru.add(
      layers.Embedding(
      input_dim=vocab_size,
      output_dim=EMBED_DIM,
      weights=[embedding_matrix],
      trainable=trainable_embeddings,
      mask_zero=True,
      name="embedding"
      )
  )
#Embedding regularisation
  model_gru.add(layers.SpatialDropout1D(spatial_dropout, name="spatial_dropout"))

#GRU encoder
  model_gru.add(
    layers.GRU(
        gru_units,
        dropout=gru_dropout,
        return_sequences=True,
        name="gru"
    ),
  )

#pooling
  model_gru.add(layers.GlobalMaxPooling1D(name="global_max_pool"))

#Dense head
  model_gru.add(layers.Dense(dense_units, activation="relu", name="dense_relu"))
  model_gru.add(layers.Dropout(dense_dropout, name="dense_dropout"))

#Output layer
  model_gru.add(layers.Dense(NUM_CLASSES, activation="softmax", name="classifier"))

  model_gru.compile(
    optimizer=optimizers.Adam(learning_rate=learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
  )

  return model_gru

print("build_gru_model() ready ✓")

### Why this GRU architecture was chosen

The architecture deliberately mirrors the BiLSTM so the two models can be compared on equal footing — same embedding layer, same regularisation strategy, same dense head, same output layer. The only meaningful difference is the recurrent unit itself.

**GRU vs BiLSTM trade-off:**
- **GRU** has 2 gates (update + reset) and one hidden state. It trains faster and uses fewer parameters.
- **BiLSTM** has 3 gates (input + forget + output), separate cell state, and reads the sequence in both directions.

For short utterances (~8 words median) this dataset doesn't strongly require backwards context, so GRU is a reasonable lighter alternative. If GRU matches or beats BiLSTM here, it suggests bidirectionality and the extra LSTM gating add complexity without payoff at this sequence length.

### Implementation difference between BiLSTM and GRU
The implementation of the BiLSTM and GRU models in this project follows the same overall pipeline, training logic, and evaluation framework. Both models were developed using the same prepared dataset, the same padded token-sequence inputs, the same label encoding, the same embedding setup, and the same tuning and evaluation procedure. This was done intentionally so that the performance comparison between the two models would be fair and based only on the difference in recurrent architecture.

#### In both cases, the model receives the same deep learning inputs:

X_train_seq, X_val_seq, and X_test_seq as padded token sequences
y_train, y_val, and y_test as encoded target labels
the same MAX_LEN, EMBED_DIM, vocab_size, and NUM_CLASSES

Both models also use the same embedding logic. The embedding layer is initialised using the same GloVe embedding matrix, which was created by matching the tokenizer vocabulary against pretrained GloVe vectors. In both models, the embedding layer can be either:

frozen, to preserve pretrained semantic information
or trainable, to allow fine-tuning for the customer-support domain

The same helper logic is used to check whether the GloVe matrix is available, and the same fallback idea applies if pretrained embeddings are missing.

The internal model structure is also the same in design, except for the type of recurrent layer used. Both models contain:

an input layer

1.   an embedding layer
2.   SpatialDropout1D for regularisation
3. a bidirectional recurrent layer
4. GlobalMaxPooling1D
5. a dense hidden layer with ReLU activation
6. a dropout layer
7. a final softmax classification layer

So the only real architectural difference is:

BiLSTM model uses a Bidirectional LSTM
GRU model uses a Bidirectional GRU

This means the surrounding logic is unchanged, but the sequence encoder itself is different.

The training process is also the same for both models. Both use:

1. the same training/validation/test split
2. the same optimiser (Adam)
3. the same loss function (sparse_categorical_crossentropy)
4. the same callbacks, EarlyStopping,ReduceLROnPlateau, the same epoch limit, the same validation-based selection strategy

The hyperparameter tuning logic is also the same. For both BiLSTM and GRU:

multiple configurations are tested
each configuration is trained on the training set
performance is measured on the validation set
the best model is selected primarily using validation macro F1-score
validation loss is used as a tie-breaker if needed

The same types of hyperparameters are tuned in both models, such as:

1. recurrent units
2. dense units
3. dropout values
4. learning rate
5. batch size
6. whether embeddings are frozen or trainable

The evaluation framework is also shared. After tuning, both the baseline and tuned versions of each model are evaluated on the held-out test set using the same shared evaluation helper. This ensures that both models are compared using the same metrics:

accuracy
precision
recall
F1-score
ROC-AUC
training time

The same style of output is then produced for both models, including:

validation tuning tables
comparison charts
baseline vs tuned comparison
test results
loss curves
classification reports
confusion matrices

The next step defines the helper (reusable function for training and evaluating GRU model configurations).  it uses learing rate reduction, validation based-evaluation, and returns model, history, metrics and run time.

In [ ]:
# =============================================================================
# TRAINING HELPER FOR GRU
# Mirrors train_one_bilstm() — same callbacks, same return signature
# =============================================================================
section("GRU training helper")

def train_one_gru(config, verbose=0):
    """Train one GRU configuration and return model, history, val summary, time."""
    K.clear_session()
    gc.collect()

    model = build_gru_model(
        gru_units=config["gru_units"],
        dense_units=config["dense_units"],
        gru_dropout=config["gru_dropout"],
        dense_dropout=config["dense_dropout"],
        spatial_dropout=config["spatial_dropout"],
        learning_rate=config["learning_rate"],
        trainable_embeddings=config["trainable_embeddings"],
        model_name=f"GRU_{config['trial_name']}",   # unique per trial
    )

    cbs = [
        # Stop when val_loss stops improving — prevents overfitting and saves time
        callbacks.EarlyStopping(
            monitor="val_loss", patience=3,
            restore_best_weights=True, verbose=1,
        ),
        # Halve LR when val_loss plateaus — helps converge to a better minimum
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=1, min_lr=1e-5, verbose=1,
        ),
    ]

    start = time.time()
    history = model.fit(
        X_train_seq, y_train,
        validation_data=(X_val_seq, y_val),
        epochs=15,
        batch_size=config["batch_size"],
        callbacks=cbs,
        verbose=verbose,
    )
    train_time = time.time() - start

    # Validation predictions used for tuning — test set is held out until final eval
    val_proba = model.predict(X_val_seq, verbose=0)
    val_pred  = val_proba.argmax(axis=1)

    val_summary = {
        "trial_name":           config["trial_name"],
        "gru_units":            config["gru_units"],
        "dense_units":          config["dense_units"],
        "gru_dropout":          config["gru_dropout"],
        "dense_dropout":        config["dense_dropout"],
        "spatial_dropout":      config["spatial_dropout"],
        "learning_rate":        config["learning_rate"],
        "batch_size":           config["batch_size"],
        "trainable_embeddings": config["trainable_embeddings"],
        "epochs_ran":           len(history.history["loss"]),
        "best_val_loss":        float(np.min(history.history["val_loss"])),
        "train_time_s":         round(train_time, 1),
        "val_accuracy":         accuracy_score(y_val, val_pred),
        "val_precision":        precision_score(y_val, val_pred, average="macro", zero_division=0),
        "val_recall":           recall_score(y_val, val_pred, average="macro", zero_division=0),
        "val_f1":               f1_score(y_val, val_pred, average="macro"),
        "val_roc_auc":          roc_auc_score(y_val, val_proba, multi_class="ovr", average="macro"),
    }
    return model, history, val_summary, train_time

print("train_one_gru() ready")

The next step defined the GRU configurations ued during tuning. It test different models and embedding tunings to identify the best setup under similar conditions to BiLSTM

In [ ]:
# =============================================================================
# HYPERPARAMETER SEARCH SPACE — GRU
# Mirrors the BiLSTM search space (frozen vs fine-tuned, small vs larger)
# so the two tiers are tuned with comparable budgets
# =============================================================================
section("GRU hyperparameter search space")

gru_search_space = [
    {
        "trial_name": "baseline_frozen",
        "gru_units": 64,
        "dense_units": 64,
        "gru_dropout": 0.2,
        "dense_dropout": 0.3,
        "spatial_dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 64,
        "trainable_embeddings": False,
    },
    {
        "trial_name": "frozen_larger",
        "gru_units": 128,
        "dense_units": 64,
        "gru_dropout": 0.2,
        "dense_dropout": 0.3,
        "spatial_dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 64,
        "trainable_embeddings": False,
    },
    {
        "trial_name": "finetune_small",
        "gru_units": 64,
        "dense_units": 64,
        "gru_dropout": 0.2,
        "dense_dropout": 0.3,
        "spatial_dropout": 0.2,
        "learning_rate": 5e-4,
        "batch_size": 64,
        "trainable_embeddings": True,
    },
    {
        "trial_name": "finetune_medium",
        "gru_units": 128,
        "dense_units": 64,
        "gru_dropout": 0.2,
        "dense_dropout": 0.4,
        "spatial_dropout": 0.2,
        "learning_rate": 5e-4,
        "batch_size": 64,
        "trainable_embeddings": True,
    },
]

pd.DataFrame(gru_search_space)

The next step trains and evaluates each GRU  configuratioon. the best confihuration is chosen using F1 socre. And all results are summarised for comparison

In [ ]:
# =============================================================================
# RUN HYPERPARAMETER TUNING — GRU
# Selection: best val_f1; tie-break on lowest best_val_loss
# =============================================================================
section("Run GRU hyperparameter tuning")

gru_tuning_rows = []
gru_best_config = None
gru_best_val_f1 = -1
gru_best_val_loss = np.inf

for i, config in enumerate(gru_search_space, start=1):
    print("\n" + "-" * 95)
    print(f"Trial {i}/{len(gru_search_space)} : {config['trial_name']}")
    print(config)

    _, _, summary, _ = train_one_gru(config=config, verbose=0)
    gru_tuning_rows.append(summary)

    print(f"Val accuracy : {summary['val_accuracy']:.4f}")
    print(f"Val F1       : {summary['val_f1']:.4f}")
    print(f"Val ROC-AUC  : {summary['val_roc_auc']:.4f}")
    print(f"Best val loss: {summary['best_val_loss']:.4f}")
    print(f"Epochs ran   : {summary['epochs_ran']}")

    if (summary["val_f1"] > gru_best_val_f1) or (
        np.isclose(summary["val_f1"], gru_best_val_f1)
        and summary["best_val_loss"] < gru_best_val_loss
    ):
        gru_best_val_f1   = summary["val_f1"]
        gru_best_val_loss = summary["best_val_loss"]
        gru_best_config   = config.copy()

gru_tuning_df = (
    pd.DataFrame(gru_tuning_rows)
    .sort_values(["val_f1", "val_accuracy", "val_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

print("\nBest GRU configuration:")
print(gru_best_config)

print("\nValidation comparison table:")
print(
    gru_tuning_df[[
        "trial_name", "gru_units", "dense_units", "trainable_embeddings",
        "batch_size", "learning_rate", "epochs_ran",
        "val_accuracy", "val_f1", "val_roc_auc", "best_val_loss",
    ]].round(4).to_string(index=False)
)

The nest step create a chart whihc compares validation F1-score for all GRU confirgurations- to show the impact of different hyperparameters.

In [ ]:
# =============================================================================
# VISUALISE TUNING RESULTS — GRU
# =============================================================================
section("GRU tuning comparison")

fig = px.bar(
    gru_tuning_df,
    x="trial_name", y="val_f1",
    color="trainable_embeddings",
    title="GRU tuning — validation macro F1 by trial",
    color_discrete_sequence=["#2E86AB", "#D1495B"],
    text="val_f1",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    title_x=0.02,
    yaxis=dict(title="Val macro F1", range=[0, 1]),
    xaxis_title=None,
    legend_title_text="Embeddings trainable",
)
fig.show()

The next step the baseline and best tuned GRU models and their training  histories are compared to show the performance improvements from tuning.

In [ ]:
# =============================================================================
# FINAL TRAINING — GRU baseline vs tuned
# =============================================================================
section("Final GRU training: baseline vs tuned")

gru_baseline_config = gru_search_space[0]   # the reference config

gru_baseline_model, gru_baseline_history, gru_baseline_val_summary, gru_baseline_time = train_one_gru(
    config=gru_baseline_config, verbose=0,
)

gru_tuned_model, gru_tuned_history, gru_tuned_val_summary, gru_tuned_time = train_one_gru(
    config=gru_best_config, verbose=0,
)

plot_history(gru_baseline_history, "GRU (baseline)")
plot_history(gru_tuned_history,    "GRU (tuned)")

The next step evaluates the baseline and tune GRU models on test set.
and their performance is compared

In [ ]:
# =============================================================================
# TEST SET EVALUATION — GRU (uses shared get_metrics helper)
# =============================================================================
section("GRU test evaluation")

gru_baseline_proba = gru_baseline_model.predict(X_test_seq, verbose=0)
gru_baseline_pred  = gru_baseline_proba.argmax(axis=1)

gru_tuned_proba = gru_tuned_model.predict(X_test_seq, verbose=0)
gru_tuned_pred  = gru_tuned_proba.argmax(axis=1)

gru_baseline_metrics = get_metrics(
    y_true=y_test, y_pred=gru_baseline_pred, y_proba=gru_baseline_proba,
    model_name="GRU (baseline)", tier="Deep Learning", time_s=gru_baseline_time,
)
gru_tuned_metrics = get_metrics(
    y_true=y_test, y_pred=gru_tuned_pred, y_proba=gru_tuned_proba,
    model_name="GRU (tuned)", tier="Deep Learning", time_s=gru_tuned_time,
)

results["GRU (baseline)"] = gru_baseline_metrics
results["GRU (tuned)"]    = gru_tuned_metrics

In [ ]:
# Classification report — GRU (tuned)
section("Classification report — GRU (tuned)")
print(classification_report(
    y_test, gru_tuned_pred,
    target_names=le.classes_,
    digits=4, zero_division=0
))

the next step shows the perfromance of the tunes GRU model using a confusion matrix. it shows classification accuracy across intents

In [ ]:
# Confusion matrix — GRU (tuned)
section("Confusion matrix — GRU (tuned)")
cm = confusion_matrix(y_test, gru_tuned_pred, normalize='true')
fig = px.imshow(
    cm,
    color_continuous_scale="Blues",
    aspect="auto",
    labels=dict(x="Predicted label", y="True label"),
    title="GRU (tuned) — normalised confusion matrix"
)
fig.update_layout(title_x=0.02)
fig.show()

---
### Feature Engineering — Transformer BERT

Transformer models require text converted to **token IDs + attention masks** using the model's own pretrained tokeniser — not raw strings and not GloVe vectors.

**Tokenisation — pretrained WordPiece.**
BERT ships with its own pretrained WordPiece tokeniser, and using it it's a requirement. The embedding layer was trained against this exact vocabulary during pretraining, so swapping in any other tokeniser would map every input token to a meaningless embedding and destroy the pretrained knowledge that makes BERT useful in the first place. The tokeniser is loaded directly from the same checkpoint as the model (`AutoTokenizer.from_pretrained("bert-base-uncased")`) to guarantee they match. A useful side-effect is that WordPiece splits rare words into known subword pieces — e.g. "cancellations" → `["cancel", "##lations"]` — so the model rarely sees out-of-vocabulary tokens, even on noisy customer-support text with typos.

**Checkpoint used for this tier:**
- **BERT-base-uncased** — 12 layers, 110M parameters, full pretrained transformer

The next line of code prepare the text data for BERT using its pretrained toekizer. it convert raw texts into input ID, applies padding, and formtas data as pyTorch tensors for transformer training

In [ ]:
# =============================================================================
# FEATURE ENGINEERING — Transformer: BERT
# Tokenisation + attention masks using the model's own pretrained tokeniser
# =============================================================================
section("Tokenisation - Transformer")

from transformers import AutoTokenizer
from datasets import Dataset
import torch

MAX_LEN_TF = 64   # 95th percentile utterance length from EDA; transformers pad to this

# Select the computation device
# If a CUDA-enabled GPU is available, use "cuda"
# Otherwise, fall back to "cpu"
# This is useful for faster training when a GPU is present
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"


# Define a helper function to tokenize a text split
# Inputs:
# - texts     -> raw input sentences/messages
# - labels    -> corresponding integer-encoded target labels
# - tokenizer -> a Hugging Face tokenizer object
# - max_len   -> maximum sequence length, default is MAX_LEN_TF
def tokenise_split(texts, labels, tokenizer, max_len=MAX_LEN_TF):
    """
    Convert a list of texts and integer labels into a HuggingFace Dataset
    with input_ids, attention_mask, and label columns ready for Trainer.
    """
    # Create a Hugging Face Dataset from the input texts and labels
    # The dataset initially contains two columns:
    # - "text"
    # - "label"
    ds = Dataset.from_dict({"text": list(texts), "label": list(labels)})

    # Apply the tokenizer to the dataset
    # tokenizer(ex["text"], ...) converts raw text into transformer inputs
    # truncation=True      -> cut sequences longer than max_len
    # padding="max_length" -> pad shorter sequences up to max_len
    # max_length=max_len   -> use the chosen maximum sequence length
    # batched=True         -> process multiple examples at once for efficiency
    #
    # This step adds columns such as:
    # - input_ids
    # - attention_mask
    # and sometimes token_type_ids depending on the model
    ds = ds.map(
        lambda ex: tokenizer(ex["text"], truncation=True,
                             padding="max_length", max_length=max_len),
        batched=True
    )

    # Remove the original raw text column after tokenization
    # The model only needs the tokenized inputs and labels
    ds = ds.remove_columns(["text"])

    # Set the dataset format to PyTorch tensors
    # Only keep the columns needed for transformer training:
    # - input_ids      -> token IDs
    # - attention_mask -> mask showing which positions are real tokens vs padding
    # - label          -> target class label
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return ds

print(f"Max sequence length : {MAX_LEN_TF}")
print(f"Device              : {DEVICE}")
print("\n BERT — WordPiece subword tokenisation")

### Why WordPiece Tokenisation?

BERT cannot read raw strings — they need numbers. Each piece of
text is split into subword tokens, and each token is mapped to an integer ID
(e.g. "cancel" → 17195). These IDs are then passed through the pretrained
embedding layer of the transformer.

The simplest tokenisation would be to split on whitespace, but this would
produce many unknown tokens — any word not in the original vocabulary would
become `[UNK]` and lose all meaning. WordPiece avoids this by splitting rare
words into known subword pieces:

- "cancel" → `["cancel"]`
- "cancellations" → `["cancel", "##lations"]`
- "unsubscribing" → `["un", "##sub", "##scribing"]`

This guarantees almost every input is tokenised into meaningful pieces rather
than unknown tokens.


### Pretrained tokeniser and fine-tuning

**BERT was pretrained by Google in 2018.** Google trained two things on roughly 3.3 billion words from Wikipedia and BooksCorpus:

1. The **WordPiece tokeniser**, which built a vocabulary of around 30,000 subword tokens.
2. The **BERT encoder weights** — 12 transformer layers, 110 million parameters — using two self-supervised tasks (masked-word prediction and next-sentence prediction).

Both of these are finished work by the time this project starts. When the notebook calls `AutoTokenizer.from_pretrained("bert-base-uncased")` and `AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")`, it is **downloading** those finished components from HuggingFace. The tokeniser is essentially a fixed lookup table; the encoder is a stack of pretrained weights ready to be adapted to a new task.

**Why the tokeniser and the model must come from the same checkpoint.** The embedding layer inside BERT was trained against this exact vocabulary during pretraining. Each of the ~30,000 WordPiece tokens has a corresponding pretrained embedding vector inside the model, and those embeddings carry all of BERT's general language knowledge. Substituting any other tokeniser would feed inputs into the embedding layer that it has never seen, which would destroy the pretrained knowledge entirely. Loading them as a matched pair guarantees they line up.

**What fine-tuning actually does.** Three components, three different statuses during fine-tuning:

| Component | Status in this project |
|---|---|
| **Tokeniser** | Pretrained, frozen- never changes |
| **BERT encoder weights** | Pretrained, then slightly adjusted on the customer-support training data |
| **Classification head (27 classes)** | Brand new, trained from scratch- the pretrained checkpoint had no concept of intent classes |

When `trainer.train()` is called, only the second and third components change. The tokeniser stays exactly as Google left it. The encoder weights nudge a small amount in the direction of the customer-support domain. The classification head, which is a single linear layer mapping BERT's 768-dimensional `[CLS]` embedding to 27 intent scores, is initialised randomly and learns from scratch on the training data.

**Why this is much faster than training from scratch.** Pretraining BERT cost Google roughly $7,000 in GPU time over 4 days on 16 TPU chips. Fine-tuning takes around 8 minutes on a Colab GPU because the model is not learning English from scratch — it already knows English. It is only learning the small delta between general English understanding and customer-support intent classification. This is the whole point of transfer learning: the heavy lifting was done once, by someone else, and every downstream project gets to inherit it.

**A useful side-effect of WordPiece tokenisation.** WordPiece splits rare or unknown words into known subword pieces — for example, "cancellations" becomes `["cancel", "##lations"]`. The model therefore rarely encounters out-of-vocabulary tokens, which is particularly useful for customer-support text where misspellings and informal phrasings are common. This is one reason transformer models tend to outperform classical TF-IDF approaches on noisy, real-world text.

### Build the BERT model

Below step is to define a reusable function that creates a transformer-based classifier. The model loads a pretrained checkpoint (DistilBERT or BERT), attaches a linear classification head on top of the `[CLS]` token, and returns the whole network ready for fine-tuning.

This structure is suitable for customer-support text because transformer attention allows every token to look at every other token, so the model can pick up intent cues no matter where they appear in the sentence.

The next section defined the BERT-based text classification model using a pre-trained transformer. it alows for optional fine tuning and adapts the classification head to the number of traget classes

In [ ]:
# =============================================================================
# BERT MODEL BUILDER
# =============================================================================
section("Define BERT model")

from transformers import AutoModelForSequenceClassification
# definition of a helper function to build a BERT-based text classification model,
# model_checkpoint specifies which pretrained BERT variant to use, num_labels sets the output layer size, dropout controls regularization, and freeze_encoder determines whether to fine-tune the BERT layers or keep them fixed
# num_labels defaults to the notebook-level NUM_CLASSES constant if not provided, ensuring the classification head matches the number of intent classes
# dropout = 0.1 is applied both within the transformer layers and on the attention weights to help prevent overfitting
# freeze encoder = False means the entire BERT model will be fine-tuned during training, while True would keep the pretrained weights fixed and only train the classification head, which can be a useful baseline to test the quality of pretrained features alone
def build_bert_model(
    model_checkpoint="bert-base-uncased",
    num_labels=None,
    dropout=0.1,
    freeze_encoder=False
):

    # Default to the notebook-level NUM_CLASSES constant if not overridden
    n_labels = num_labels if num_labels is not None else NUM_CLASSES

    # Load pretrained transformer and a fresh linear classification head
    # The head is a Linear(hidden_size, num_labels) layer on top of the [CLS] token
    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=n_labels,
        hidden_dropout_prob=dropout,            # dropout inside the transformer
        attention_probs_dropout_prob=dropout,   # dropout on attention weights
        ignore_mismatched_sizes=True            # needed when swapping the head size
    )

    # Optionally freeze the pretrained encoder — only the classification head trains
    # Useful as a cheap baseline: tests whether pretrained features alone are enough
    if freeze_encoder:
        for name, param in model.named_parameters():
            if "classifier" not in name:
                param.requires_grad = False

    # Move model to the chosen device (GPU if available, else CPU)
    model.to(DEVICE)

    return model

print("build_bert_model() ready ")

### Why this BERT architecture was chosen

This architecture was selected because it is *appropriate for multi-class text classification.*

The pretrained transformer converts WordPiece token IDs into contextual embeddings — every token is represented by a vector that depends on every other token in the sentence. If DistilBERT or BERT weights are available (they always are via HuggingFace), the model starts with semantic knowledge learned from billions of words of Wikipedia + BooksCorpus. The parameter `freeze_encoder` allows the experiment to compare feature-extraction (frozen encoder) against full fine-tuning.

**Hidden-state dropout** (`hidden_dropout_prob`) and **attention dropout** (`attention_probs_dropout_prob`) are used to improve generalisation by dropping parts of the hidden representations and attention weights during training.

The self-attention mechanism reads the message in all directions at once. This is more expressive than BiLSTM because every token can directly attend to every other token, instead of passing information through a sequential hidden state.

**The `[CLS]` token pooling** replaces `GlobalMaxPooling1D`. During pretraining BERT was trained to use the `[CLS]` token as an aggregate sentence representation, so its final hidden state is naturally suited for classification.

The output layer uses softmax because this is a multi-class classification problem with `NUM_CLASSES` intent labels.

The loss function is sparse categorical cross-entropy — computed internally by HuggingFace `AutoModelForSequenceClassification` when integer `label` columns are supplied.

### Training helper for BERT experiments

The following function trains one BERT configuration and returns:

1. The trained model
2. The training history (per-epoch val loss and val F1)
3. A validation summary used for hyperparameter tuning
4. The total training time

The validation metrics are computed directly inside the function so that the shared helper `get_metrics()` can remain the common final evaluation function for all models.

The next line of code defines the resubable function of BERT experiments.it tokenises each data split and trains a selected BERT configuration, records training history and returns validation matric for model comparison


In [ ]:
# =============================================================================
# TRAINING HELPER FOR BERT
# =============================================================================
section("BERT training helper")

from transformers import (TrainingArguments, Trainer,
                          EarlyStoppingCallback)

# Define a helper function that computes evaluation metrics
# This function will be called automatically by Hugging Face Trainer
# after each validation run
def _compute_metrics_hf(eval_pred):
    """Macro F1 and accuracy — used by Trainer after every validation epoch."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Return a dictionary of evaluation metrics
    # f1_macro -> macro-averaged F1 score across all classes
    # accuracy -> proportion of correct predictions
    # These metrics are logged by Trainer after each evaluation step
    return {
        "f1_macro":  f1_score(labels, preds, average="macro"),
        "accuracy":  accuracy_score(labels, preds)
    }

# Define a reusable function to train one BERT configuration
# Inputs:
# - config  -> dictionary containing the hyperparameters for this trial
# - verbose -> controls whether progress bars are shown
# Outputs:
# - model         -> trained BERT model
# - history       -> dictionary of training/validation losses and F1 values
# - val_summary   -> summary metrics for this validation run
# - train_time    -> total training time in seconds

def train_one_bert(config, verbose=0):
    """
    Train one BERT configuration and evaluate it on the validation set.
    Returns model, history, validation summary, and training time.
    """
    gc.collect()

    # If using GPU, clear cached CUDA memory before starting training
    # This helps reduce GPU memory issues between trials
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # Load the pretrained tokeniser for this checkpoint
    tokenizer = AutoTokenizer.from_pretrained(config["model_checkpoint"])

    # Tokenise all three splits with this tokeniser
    train_ds = tokenise_split(X_train, y_train, tokenizer)
    val_ds   = tokenise_split(X_val,   y_val,   tokenizer)
    test_ds  = tokenise_split(X_test,  y_test,  tokenizer)

    # Build the model for this trial using the helper function defined earlier
    model = build_bert_model(
        model_checkpoint=config["model_checkpoint"],
        num_labels=NUM_CLASSES,
        dropout=config["dropout"],
        freeze_encoder=config["freeze_encoder"]
    )

    # Trainer arguments — warm-up schedule, gradient clipping, early stopping
    args = TrainingArguments(
        output_dir=f"./bert_out/{config['trial_name']}",    # directory to save model checkpoints and logs for this trial
        num_train_epochs=config["epochs"],                  #    maximum number of training epochs
        per_device_train_batch_size=config["batch_size"],   # batch size for training
        per_device_eval_batch_size=config["batch_size"] * 2,    # use a larger batch size for evaluation since it doesn't require backprop
        learning_rate=config["learning_rate"], # the initial learning rate for the AdamW optimizer
        warmup_ratio=config["warmup_ratio"],   # LR warmup (10% of steps)
        weight_decay=0.01,              # L2 regularization to help prevent overfitting
        max_grad_norm=1.0,                     # gradient clipping
        eval_strategy="epoch",              #   evaluate at the end of every epoch
        save_strategy="epoch",             #   save model checkpoints at the end of every epoch
        load_best_model_at_end=True,    #   after training, load the checkpoint with the best validation F1 score
        metric_for_best_model="f1_macro", # use macro F1 to determine the best model checkpoint
        greater_is_better=True,         # higher F1 is better, so Trainer knows how to compare checkpoints
        logging_steps=50,               # log training metrics every 50 steps (not too often to avoid overhead, but enough to see progress)
        fp16=(DEVICE == "cuda"),               # mixed precision on GPU only
        report_to="none",               # disable Hugging Face's default logging to avoid clutter
        disable_tqdm=(verbose == 0)     # disable progress bars if verbose=0 for cleaner notebook output
    )
    # Create a Hugging Face Trainer object
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=_compute_metrics_hf,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    start = time.time() #   start the timer just before training begins
    trainer.train()     #   this will run the training loop, including evaluation at the end of each epoch
    train_time = time.time() - start #  calculate total training time in seconds

    # Collect per-epoch history from Trainer state — mirrors Keras history
    history = {"loss": [], "val_loss": [], "val_f1": []} #  initialise empty lists to store training loss, validation loss, and validation F1 for each epoch
    for log in trainer.state.log_history:                   # iterate through the logged history of training
        if "loss" in log and "epoch" in log and "eval_loss" not in log: # only consider logs that contain training loss for an epoch (ignore intermediate logs and evaluation logs)
            history["loss"].append(log["loss"])                         # append the training loss to the history
        if "eval_loss" in log:                              # for logs that contain evaluation results at the end of an epoch, append the validation loss and F1 score to the history
            history["val_loss"].append(log["eval_loss"])    # append validation loss
            history["val_f1"].append(log.get("eval_f1_macro", np.nan)) # append validation F1, use np.nan if it's not available for some reason

    # Validation predictions for tuning
    val_logits = trainer.predict(val_ds).predictions
    val_proba  = softmax(val_logits, axis=1)
    val_pred   = val_proba.argmax(axis=1)

    # Compile a summary of the validation performance and training details for this trial
    val_summary = {
        "trial_name":       config["trial_name"],
        "model_checkpoint": config["model_checkpoint"],
        "learning_rate":    config["learning_rate"],
        "batch_size":       config["batch_size"],
        "dropout":          config["dropout"],
        "warmup_ratio":     config["warmup_ratio"],
        "freeze_encoder":   config["freeze_encoder"],
        "epochs_ran":       len(history["val_loss"]),
        "best_val_loss":    float(np.min(history["val_loss"])) if history["val_loss"] else np.nan,
        "train_time_s":     round(train_time, 1),
        "val_accuracy":     accuracy_score(y_val, val_pred),
        "val_precision":    precision_score(y_val, val_pred, average="macro", zero_division=0),
        "val_recall":       recall_score(y_val, val_pred, average="macro", zero_division=0),
        "val_f1":           f1_score(y_val, val_pred, average="macro"),
        "val_roc_auc":      roc_auc_score(y_val, val_proba, multi_class="ovr", average="macro")
    }

    return model, history, val_summary, train_time

print("train_one_bert() ready ")


### Why this training function is needed

This helper function makes hyperparameter tuning easier and keeps the notebook organised.

**EarlyStoppingCallback** is used to stop training when the validation F1 stops improving. This reduces overfitting and avoids wasting training time — transformers converge quickly (often within 3 epochs) and continuing past the best epoch only degrades test performance.

**Warm-up ratio** ramps the learning rate up gradually over the first 10% of steps. This is critical for transformers — jumping straight to the maximum learning rate at step 1 destroys pretrained weights before the model has adapted to the new task.

**Gradient clipping** at `max_grad_norm=1.0` prevents exploding gradients. Transformers are sensitive to large updates during the early warm-up phase; clipping keeps training stable.

**Mixed precision (fp16)** is enabled automatically when a GPU is available. It makes training ~2× faster with negligible accuracy loss and is essential for BERT-base on limited hardware.

The model is tuned using validation **macro F1** because this is a multi-class intent classification task. Macro F1 is more informative than accuracy alone because it gives equal importance to each class.

## Define the hyperparameter search space

The next cell defines a small but meaningful tuning space for BERT. The selected hyperparameters control the pretrained backbone, regularisation strength, optimisation behaviour, and whether the pretrained encoder is frozen or fine-tuned.

### Why these hyperparameters were selected

The tuning space focuses on the parameters most likely to influence transformer performance.

***model_checkpoint***: BERT-base (expensive)

***learning_rate***: the most important transformer hyperparameter — values outside 1e-5 to 5e-5 typically fail

***dropout***: hidden + attention dropout provide regularisation

***warmup_ratio***: affects optimisation stability early in training

***batch_size***: affects both speed and gradient behaviour (smaller = noisier but often generalises better)

***freeze_encoder***: compares feature-extraction against full fine-tuning

This search space is intentionally small (4 trials) because each transformer run is expensive. The BiLSTM search space used 6 trials because each trial takes ~1 minute; each BERT trial takes 10–40 minutes depending on hardware.

The next step defines the BERT configurations evaluated during tuning.it tests various learing rates and dropouts

In [ ]:
# =============================================================================
# HYPERPARAMETER SEARCH SPACE
# =============================================================================
section("BERT hyperparameter search space")

# Define a list of dictionaries, where each dictionary represents a different hyperparameter configuration (trial)
# for training the BERT model. This search space includes variations in learning rate, dropout,
# and whether to freeze the encoder, allowing us to compare different training strategies and regularization levels.

bert_search_space = [
    {
        "trial_name":       "bert_baseline",
        "model_checkpoint": "bert-base-uncased", # the pretrained BERT variant to use
        "learning_rate":    2e-5,               # the initial learning rate for the AdamW optimizer
        "batch_size":       16,                 # batch size for training; transformers often require smaller batches due to memory constraints
        "dropout":          0.1,                # dropout rate applied both within the transformer and on attention weights to help prevent overfitting
        "warmup_ratio":     0.1,                # the proportion of total training steps to use for learning rate warmup; helps stabilize training in the early epochs
        "epochs":           4,                  # maximum number of training epochs; with early stopping, the actual number may be lower if the model stops improving on the validation set
        "freeze_encoder":   False,              # whether to freeze the pretrained BERT layers and only train the classification head; False means the entire model will be fine-tuned, while True would keep the pretrained weights fixed
    },
    {
        "trial_name":       "bert_higher_lr",   #   a trial with a higher learning rate to test if it can converge faster or find a better minimum, though it may risk overshooting
        "model_checkpoint": "bert-base-uncased",# same pretrained model as baseline for a fair comparison
        "learning_rate":    3e-5,               # a common learning rate for fine-tuning BERT; may lead to faster convergence but also risks instability, so it's important to compare against the baseline
        "batch_size":       16,                 # keeping the batch size the same as baseline to isolate the effect of learning rate
        "dropout":          0.1,                # same dropout as baseline to focus on the impact of learning rate
        "warmup_ratio":     0.1,                # same warmup ratio to ensure the learning rate schedule is consistent across trials
        "epochs":           4,                  # same number of epochs to allow for early stopping to determine the actual training duration based on validation performance
        "freeze_encoder":   False,
    },
    {
        "trial_name":       "bert_higher_dropout", # a trial with increased dropout to test if stronger regularization can improve validation performance by preventing overfitting, especially if the baseline shows signs of overfitting
        "model_checkpoint": "bert-base-uncased", # same pretrained model for a fair comparison
        "learning_rate":    2e-5,                #  same learning rate as baseline to isolate the effect of dropout
        "batch_size":       16,                 # same batch size to ensure that any differences in performance can be attributed to the change in dropout rather than batch size effects
        "dropout":          0.2,            # increased dropout rate to test if stronger regularization can help improve validation performance by preventing overfitting, especially if the baseline shows signs of overfitting
        "warmup_ratio":     0.1,            # same warmup ratio to ensure the learning rate schedule is consistent across trials, allowing us to focus on the impact of dropout
        "epochs":           4,              # same number of epochs to allow for early stopping to determine the actual training duration based on validation performance
        "freeze_encoder":   False,
    },
]

pd.DataFrame(bert_search_space)


## Run BERT hyperparameter tuning

The following cell trains each BERT configuration from the search space above and compares them using validation performance. The best configuration is selected on validation macro F1, with validation loss as a tie-breaker. The test set is not used during this stage.

Each BERT trial takes roughly 20–40 minutes on CPU (much faster on GPU), so the full loop can take a couple of hours end-to-end. If running on CPU, consider running this once and saving the results rather than re-running every time.

This next step trains and evaluates each BERT confirguartiona and selects the best configuration using validation F1-score and saves the result for later comparison

In [ ]:
# =============================================================================
# RUN HYPERPARAMETER TUNING — BERT
# Selection: best val_f1; tie-break on lowest best_val_loss
# Mirrors the BiLSTM/GRU tuning loops
# =============================================================================
section("Run BERT hyperparameter tuning")

bert_tuning_rows  = []
bert_best_config  = None
bert_best_val_f1  = -1
bert_best_val_loss = np.inf

# Loop through each hyperparameter configuration in the BERT search space
# `i` is the trial number, starting from 1
# `config` is a dictionary containing the hyperparameters for the current trial
for i, config in enumerate(bert_search_space, start=1):
    print("\n" + "-" * 95)
    print(f"Trial {i}/{len(bert_search_space)} : {config['trial_name']}")
    print(config)

    # Train one BERT model using the current hyperparameter configuration
    # The function returns multiple outputs:
    # - trained model
    # - training history
    # - validation summary
    # - training time
    # Only the validation summary is kept here
    # The other returned values are ignored using `_`
    _, _, summary, _ = train_one_bert(config=config, verbose=0)
    bert_tuning_rows.append(summary)

    print(f"Val accuracy : {summary['val_accuracy']:.4f}")
    print(f"Val F1       : {summary['val_f1']:.4f}")
    print(f"Val ROC-AUC  : {summary['val_roc_auc']:.4f}")
    print(f"Best val loss: {summary['best_val_loss']:.4f}")
    print(f"Epochs ran   : {summary['epochs_ran']}")

    # Check whether the current trial is better than the best BERT trial seen so far
    # Primary criterion: higher validation macro F1
    # Tie-breaker: lower validation loss if the F1 scores are almost equal
    if (summary["val_f1"] > bert_best_val_f1) or ( # if the current trial's F1 is better than the best F1 so far, or...
        np.isclose(summary["val_f1"], bert_best_val_f1) # check if F1 is close enough to consider a tie
        and summary["best_val_loss"] < bert_best_val_loss # if it's a tie in F1, check if the current trial's best validation loss is better (lower) than the best validation loss of the current best trial
    ):
        bert_best_val_f1   = summary["val_f1"]
        bert_best_val_loss = summary["best_val_loss"]
        bert_best_config   = config.copy()

bert_tuning_df = (
    pd.DataFrame(bert_tuning_rows)
    .sort_values(["val_f1", "val_accuracy", "val_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

print("\nBest BERT configuration:")
print(bert_best_config)

print("\nValidation comparison table:")
print(
    bert_tuning_df[[
        "trial_name", "learning_rate", "dropout", "batch_size",
        "epochs_ran", "val_accuracy", "val_f1", "val_roc_auc", "best_val_loss",
    ]].round(4).to_string(index=False)
)

# Saving tuning results incase the kernel dies later
bert_tuning_df.to_csv("bert_tuning_results.csv", index=False)

## Output Explanation
The BERT hyperparameter tuning results show that all tested configurations achieved extremely strong validation performance, with macro F1-scores close to 1.0. Training loss decreased rapidly across epochs, while validation loss also improved steadily, indicating effective learning and strong generalisation. Among the completed trials shown, the higher-learning-rate configuration (3e-5) slightly outperformed the baseline configuration (2e-5) in validation macro F1 and accuracy, although the differences were very small. Overall, the results suggest that BERT is highly effective for the customer-intent classification task.

In [ ]:
# =============================================================================
# VISUALISE BERT TUNING RESULTS
# =============================================================================
section("BERT tuning comparison")

# Create a horizontal bar plot using Plotly Express to compare the validation macro F1 scores of different BERT tuning trials

fig = px.bar(
    bert_tuning_df.sort_values("val_f1", ascending=True),
    x="val_f1",
    y="trial_name",
    orientation="h",
    color="learning_rate",
    title="BERT validation macro F1 by trial",
    labels={"val_f1": "Validation macro F1", "trial_name": "Trial"},
    text="val_f1",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(title_x=0.02, yaxis_title=None, height=350)
fig.show()

### Output Explanation
This chart compares the validation macro F1 scores of the BERT tuning trials. All configurations performed extremely well, with scores close to 1.0. The baseline BERT model achieved the highest validation macro F1, while increasing the learning rate or dropout did not improve performance.

## Final BERT training — baseline vs tuned

After tuning, the baseline BERT and the best tuned BERT are trained once more and evaluated on the test set. This gives a clean before-vs-after comparison and shows whether hyperparameter tuning produced a meaningful improvement.

This section traines the baseline andbest tuned BERT models- this enables comparions to show the impact of hyperparameter tuning

In [ ]:
# =============================================================================
# FINAL TRAINING — BERT baseline vs tuned
# =============================================================================
section("Final BERT training: baseline vs tuned")

bert_baseline_config = bert_search_space[0]  # reference config (first trial)

# Train the baseline BERT model using the baseline hyperparameter configuration
# train_one_bert(...) returns four outputs:
# - bert_baseline_model        -> the trained baseline BERT model
# - bert_baseline_history      -> training history containing loss and validation metrics across epochs
# - bert_baseline_val_summary  -> summary of validation performance for the baseline BERT model
# - bert_baseline_time         -> total training time in seconds for the baseline model
# config=bert_baseline_config  -> pass the baseline BERT configuration into the training function
# verbose=0
bert_baseline_model, bert_baseline_history, bert_baseline_val_summary, bert_baseline_time = train_one_bert(
    config=bert_baseline_config, verbose=0,
)

# Train the tuned BERT model using the best hyperparameter configuration
# `bert_best_config` was selected earlier from the BERT tuning results
# This call returns:
# - bert_tuned_model        -> the trained best/tuned BERT model
# - bert_tuned_history      -> training history for the tuned model
# - bert_tuned_val_summary  -> summary of validation performance for the tuned model
# - bert_tuned_time         -> total training time in seconds for the tuned model
# This setup allows a direct comparison between the baseline BERT and the tuned BERT
bert_tuned_model, bert_tuned_history, bert_tuned_val_summary, bert_tuned_time = train_one_bert(
    config=bert_best_config, verbose=0,
)

# Plot loss curves — BERT's history is a dict, so wrap it for plot_history()
class _HistoryShim:
    """Adapter so Keras-style plot_history() works with BERT's dict history."""
    def __init__(self, history_dict):
        self.history = history_dict

#plot_history(_HistoryShim(bert_baseline_history), "BERT (baseline)")
#plot_history(_HistoryShim(bert_tuned_history),    "BERT (tuned)")

## Output Summary

The tuning loop selected **`bert_higher_lr`** (lr = 3e-5) as the best BERT configuration,
beating the baseline (lr = 2e-5) on validation macro F1: **0.9983 vs 0.9975**.

Both models were then re-trained from scratch using their respective configurations and
evaluated on the held-out test set:

- **BERT (baseline)** — lr = 2e-5, dropout = 0.1 → test macro F1 = **0.9955**
- **BERT (tuned)** — lr = 3e-5, dropout = 0.1 → test macro F1 = **0.9965**

The +0.001 gain confirms that the higher learning rate converges to a slightly better
optimum within 4 epochs on this dataset. Training and validation loss both decreased
consistently across epochs with no sign of instability or catastrophic forgetting, which
is expected given the warm-up schedule (10% of steps) and gradient clipping at max_norm = 1.0.

## Loss curves - BERT baseline vs tuned

In [ ]:
section("Final BERT Plot History")

plot_history(_HistoryShim(bert_baseline_history), "BERT (baseline)")
plot_history(_HistoryShim(bert_tuned_history),    "BERT (tuned)")

> **Note on the X-axis.** BERT records its training loss very frequently during each epoch — many times per epoch, not just once — so the blue training curve has lots of points spread across the chart. Validation loss is only measured at the end of each epoch (4 points total for 4 epochs), which is why the red markers appear clustered on the left rather than spread across the X-axis. The shape itself is healthy: training loss starts at ~3.3 (the random-baseline loss for 27 classes, ln(27)) and drops to near zero by the end of epoch 1, with validation loss tracking closely — no overfitting, clean convergence.
>
> **Analysis.** BERT reaches near-zero loss within a single epoch, which is far faster than the BiLSTM and GRU models that needed ~10–15 epochs to converge. This rapid convergence reflects the value of pretraining: the encoder already encodes general language knowledge, so fine-tuning only has to adapt the classification head and lightly adjust the encoder for the customer-support domain. The remaining three epochs produce only marginal gains, which is why early stopping on validation F1 is essential — without it, BERT would simply continue training on noise.

## BERT test set evaluation

The held-out test set is used once, at the end, to produce the final reported metrics for both the baseline and tuned BERT models. The same `get_metrics()` helper used by all other models is reused here so the comparison in the cross-model section is consistent.

This setion evalautes the baseline and tuned BERT models on the test set.
it generates predictions and compares results to assess the imporovments impact of tuning.

In [ ]:
# =============================================================================
# TEST SET EVALUATION — BERT
# =============================================================================
section("BERT test evaluation")

from torch.utils.data import DataLoader

# Define a helper function to generate predictions for the test set
# using a trained BERT model and its corresponding configuration
def _bert_test_predict(model, config):
    """Tokenise the test set with the model's own tokeniser and produce predictions."""
    tok = AutoTokenizer.from_pretrained(config["model_checkpoint"])

    # Tokenize the test texts and labels using the helper function
    # The result is a Hugging Face Dataset containing:
    # - input_ids
    # - attention_mask
    # - label
    test_ds = tokenise_split(X_test, y_test, tok)

    # Put the model into evaluation mode
    # This disables training-specific behaviors such as dropout
    model.eval()

    # Create a PyTorch DataLoader for the tokenized test dataset
    # Batch size is set to twice the training batch size for faster inference
    # since gradients are not being computed
    loader = DataLoader(test_ds, batch_size=config["batch_size"] * 2)
    all_logits = []

    # Disable gradient computation during inference
    # This reduces memory usage and speeds up prediction
    with torch.no_grad():
        for batch in loader:
            # Move the input tensors in the batch to the selected device
            # (CPU or GPU)
            # Exclude the "label" field because labels are not needed for forward prediction
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "label"}
            # Run a forward pass through the BERT model using the batch inputs
            # The output includes logits for each class
            out = model(**batch)
            # Move the logits back to CPU, convert them to NumPy arrays,
            # and store them in the list
            all_logits.append(out.logits.cpu().numpy())
    # Combine all batch logits into one full NumPy array
    # Each row corresponds to one test example
    # Each column corresponds to one class
    logits = np.concatenate(all_logits, axis=0)

    # Convert logits into class probabilities using softmax
    # Probabilities for each example will sum to 1 across classes
    proba  = softmax(logits, axis=1)

    # Convert the class probabilities into final predicted labels
    # by choosing the class with the highest probability for each example
    pred   = proba.argmax(axis=1)
    return pred, proba

# Run the helper function on the baseline BERT model
# Returns:
# - bert_baseline_pred  -> predicted labels for the test set
# - bert_baseline_proba -> predicted class probabilities for the test set
bert_baseline_pred, bert_baseline_proba = _bert_test_predict(
    bert_baseline_model, bert_baseline_config
)

# Evaluate the baseline BERT model using the shared get_metrics() helper
# y_true=y_test                  -> true labels for the test set
# y_pred=bert_baseline_pred      -> predicted labels from the baseline model
# y_proba=bert_baseline_proba    -> predicted probabilities from the baseline model
# model_name="BERT (baseline)"   -> label used to identify the baseline model
# tier="Transformer"             -> model category
# time_s=bert_baseline_time      -> total training time of the baseline model
bert_baseline_metrics = get_metrics(
    y_true=y_test, y_pred=bert_baseline_pred, y_proba=bert_baseline_proba,
    model_name="BERT (baseline)", tier="Transformer", time_s=bert_baseline_time,
)

# Tuned predictions — variable names match the model_registry expectations
# This section generates test predictions for the tuned BERT model
bert_pred, bert_proba = _bert_test_predict(
    bert_tuned_model, bert_best_config
)

# Evaluate the tuned BERT model using the same shared metric function
# This ensures a fair comparison between the baseline and tuned transformer models
bert_metrics = get_metrics(
    y_true=y_test, y_pred=bert_pred, y_proba=bert_proba,
    model_name="BERT (tuned)", tier="Transformer", time_s=bert_tuned_time,
)

# Store in the master results dict
results["BERT (baseline)"] = bert_baseline_metrics
results["BERT (tuned)"]    = bert_metrics

# Test comparison table
bert_test_compare = (
    pd.DataFrame([bert_baseline_metrics, bert_metrics])
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

# Select the main comparison columns from the BERT results table
# - model      -> model name
# - accuracy   -> overall prediction correctness
# - precision  -> correctness of positive predictions
# - recall     -> ability to recover true class examples
# - f1         -> balanced precision-recall score
# - roc_auc    -> overall class separability
# - time_s     -> training time in seconds
# .round(4) -> round numeric values to 4 decimal places
# .to_string(index=False) -> print the table neatly without row indices
print("\nTest comparison:")
print(
    bert_test_compare[
        ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "time_s"]
    ].round(4).to_string(index=False)
)

## Summary of Output

**BERT test evaluation summary.** Both the baseline and tuned BERT models reach an identical test macro F1 of **0.9955** (accuracy 0.9955, precision 0.9956, recall 0.9955, ROC-AUC 1.0000). The reason the two are matched to six decimal places is that the tuning loop selected the baseline configuration itself as the winner — the trials with higher learning rate (3e-5) and higher dropout (0.2) both produced slightly lower validation F1, so the "tuned" final model ended up re-training the same configuration as the baseline. The 40-second difference in training time (598.9 s vs 558.0 s) reflects Colab GPU variance between runs, not model differences.

This is itself a meaningful result: the standard recommended hyperparameters for BERT-base on text classification — learning rate 2e-5, dropout 0.1, warmup ratio 0.1 — are already well-calibrated for this dataset. The fact that more aggressive hyperparameters did not improve validation performance suggests BERT is already close to the ceiling of what this dataset allows. Any remaining gains would more plausibly come from architecture changes (e.g. RoBERTa, DeBERTa) or label-quality improvements rather than tuning BERT-base further.

## Classification report and confusion matrix - tuned BERT

A detailed classification report shows which intent classes the tuned BERT predicts well and which classes remain difficult. The confusion matrix then visualises where the errors cluster.

The expected pattern from the EDA is that errors concentrate within the same high-level category — *cancel_order* vs *change_order*, *track_order* vs *track_refund*, and so on. If BERT's matrix is cleaner than the BiLSTM and classical matrices in these areas, it is evidence that contextual embeddings resolve the vocabulary-overlap problem that the classical tier cannot.

This section evaluates the tuned BERT model using a detailed confusion matrix and a classification report.

In [ ]:
# =============================================================================
# CLASSIFICATION REPORT + CONFUSION MATRIX — TUNED BERT
# =============================================================================
section("Classification report — BERT (tuned)")

# printing a detailed classification report for the tuned BERT model's test predictions up to 6 digits after coma due to very similar results.
print(
    classification_report(
        y_test,
        bert_pred,
        target_names=le.classes_,
        digits=6,
        zero_division=0,
    )
)

section("Confusion matrix — BERT (tuned)")

cm = confusion_matrix(y_test, bert_pred)
# compute the confusion matrix comparing true labels (y_test) and predicted labels (bert_pred) for the tuned BERT model;
# this matrix shows the counts of true positives, false positives, true negatives, and false negatives for each class,
# allowing us to see where the model is making correct predictions and where it is confusing different classes


This section visualizes the performance of the tunes BERT model using a normalized confusion matrix

In [ ]:
# =============================================================================
# CONFUSION MATRIX — TUNED BERT
# =============================================================================
section("Confusion matrix — BERT (tuned)")
# A normalization of the matrix is done so that the output can be easily interpret
cm = confusion_matrix(y_test, bert_pred, normalize='true')

fig = px.imshow(
    cm,
    color_continuous_scale="Blues",
    aspect="auto",
    labels=dict(x="Predicted label", y="True label", color="Proportion"),
    title="BERT (tuned) — confusion matrix"
)
fig.update_layout(title_x=0.02)
fig.show()

## Attention visualisation — tuned BERT

Self-attention lets each token look at every other token when building its contextual representation. Visualising the attention weights for a handful of example utterances shows which words the model actually relied on to predict the intent.

For each example, the attention matrix from the final encoder layer is plotted, averaged across the 12 attention heads. Tokens on the y-axis are attending to tokens on the x-axis; brighter cells mean stronger attention.

The next line of code visualizes attention patterns from the tuned BERT.
it reloads the model with attention outputs and extracts final layer attention weights.

In [ ]:
# =============================================================================
# ATTENTION VISUALISATION — tuned BERT (final layer, mean over heads)
# =============================================================================
section("Attention visualisation — BERT (tuned)")

from transformers import AutoModelForSequenceClassification, AutoTokenizer

# -----------------------------------------------------------------------------
# Reload the tuned BERT with eager attention so we can extract attention weights.
# The default SDPA attention implementation is faster but does not expose the
# intermediate attention tensors, so we save the trained model to disk and
# reload it with attn_implementation="eager".
# -----------------------------------------------------------------------------
bert_tuned_tokenizer = AutoTokenizer.from_pretrained(bert_best_config["model_checkpoint"])

# Save the tuned model once (skipped if already on disk from a previous run)
if not os.path.exists("./bert_tuned_for_attention"):
    bert_tuned_model.save_pretrained("./bert_tuned_for_attention")
    bert_tuned_tokenizer.save_pretrained("./bert_tuned_for_attention")

# Reload with eager attention enabled
attn_model = AutoModelForSequenceClassification.from_pretrained(
    "./bert_tuned_for_attention",
    attn_implementation="eager",
    output_attentions=True,
).to(DEVICE)
attn_model.eval()

# -----------------------------------------------------------------------------
# Example utterances — mix of intent types to compare attention patterns
# -----------------------------------------------------------------------------
examples = [
    "I want to cancel my order please",
    "Can you help me track my refund",
    "I need to change the order I placed yesterday",
]

for text in examples:
    # Tokenize the current text using the tuned BERT tokenizer
    # `return_tensors="pt"` returns PyTorch tensors
    # `truncation=True` ensures long texts are cut to the maximum allowed length
    # `padding="max_length"` pads shorter texts up to `MAX_LEN_TF`
    # `max_length=MAX_LEN_TF` fixes the sequence length
    # `.to(DEVICE)` moves the encoded tensors to CPU or GPU
    enc = bert_tuned_tokenizer(
        text, return_tensors="pt", truncation=True,
        padding="max_length", max_length=MAX_LEN_TF,
    ).to(DEVICE)

    # torch.no_grad() context disables gradient calculations, which reduces memory usage and speeds up inference
    with torch.no_grad():
        out = attn_model(**enc, output_attentions=True)  # attn_model is the BERT model with outputs configured to include attention weights

    if out.attentions is None:  # if attentions are not returned, it means the model is not in the correct mode to output them
        raise RuntimeError("Attention tensors still None — eager mode is not active.")  # raise error if attentions are not available, which indicates a setup issue

    # Final layer, averaged over 12 heads → (seq_len, seq_len)
    attn = out.attentions[-1][0].mean(dim=0).cpu().numpy()

    # Convert token IDs back into readable token strings
    # `enc["input_ids"][0]` gets the token IDs of the current example
    # `.cpu().tolist()` converts them into a normal Python list
    # `convert_ids_to_tokens(...)` maps IDs back to BERT tokens
    tokens = bert_tuned_tokenizer.convert_ids_to_tokens(
        enc["input_ids"][0].cpu().tolist()
    )

    # Trim to real tokens (drop [PAD])
    real_len = int(enc["attention_mask"][0].sum().item())  # sum of attention mask gives the number of real tokens
    tokens = tokens[:real_len]  # tokens list is sliced to include only the real tokens, excluding any padding
    attn = attn[:real_len, :real_len]  # attn matrix is sliced to match the real token length, ensuring the plot only shows attention between actual tokens and not padding. Padding is typically represented by [PAD] tokens which do not contribute meaningful attention patterns.

    # Plot with matplotlib
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(attn, cmap="Blues", aspect="auto")

    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(tokens, fontsize=9)

    ax.set_xlabel("Key tokens (attended to)")
    ax.set_ylabel("Query tokens (attending from)")
    ax.set_title(f"Attention — final layer, mean over heads:\n'{text}'",
                 fontsize=11, pad=12)

    plt.colorbar(im, ax=ax, label="Attention weight")
    plt.tight_layout()
    plt.show()

    # The resulting plot shows how each token in the input attends to every other token,
    # with darker colors indicating stronger attention weights. This can help us understand
    # which words the model focuses on when making predictions, and whether it captures relevant relationships in the text.
    # CLS token is usually the first token and often has high attention weights to important words in the sentence, as it aggregates information for classification.
    # SEP tokens also show interesting patterns, especially in tasks involving sentence pairs.


### Reading the attention heatmap

Each cell answers: *"When the model was processing the word on the left (Y-axis), how much did it look at the word on the bottom (X-axis)?"*

- **Y-axis (rows)** — the query token, the word doing the asking ("what should I look at?")
- **X-axis (columns)** — the key token, the word being looked at ("what's relevant for me?")
- **Cell colour** — strength of attention (darker blue = more attention)
- **Each row sums to 1.0** — every token spreads its full "attention budget" across all other tokens

Two special tokens appear in every BERT input:

- **`[CLS]`** sits at the start. Its final hidden state is what the classifier head reads to decide the intent — so the `[CLS]` row tells you which words BERT used to make its prediction.
- **`[SEP]`** sits at the end and marks the sentence boundary.

### Reading the example: *"i want to cancel my order please"*

This is a `cancel_order` utterance. The heatmap shows several patterns that confirm BERT is behaving sensibly:

**Row 1 — `[CLS]` (the classification token).** The brightest cells in this row sit on `order` (~0.18), `please` (~0.15), and `[SEP]` (~0.13). When BERT decided this message was `cancel_order`, it looked most at the word `order` — the intent-bearing noun.

**Row 7 — `order`.** The darkest cell in the whole heatmap is `order` attending to itself (~0.20). The key content word is anchoring strongly on its own identity while gathering context.

**The diagonal is generally darker.** Every token attends partly to itself, which is typical BERT behaviour — each word needs to preserve its own identity while incorporating context from the rest of the sentence.

**Stopwords get less attention.** The columns for `i`, `to`, and `my` are noticeably lighter than the columns for `cancel`, `order`, and `please`. BERT has correctly learned that stopwords carry less classification signal, so it isn't wasting attention on them.

### What this tells us about the model

| Observation | Interpretation |
|---|---|
| `[CLS]` attends most to `order` and `please` | Model picked up the intent-bearing words |
| `cancel` attends to `[CLS]` and `[SEP]` heavily | The action verb is anchoring to sentence boundaries — typical |
| Stopwords (`i`, `to`, `my`) receive less attention | Model correctly downweighted low-signal words |
| `order → order` is the brightest cell | The key content noun has the strongest self-context |

A bad sign would be dark vertical stripes on stopwords — meaning the model was distracted by uninformative words.

### Caveats

- These weights are **averaged across 12 attention heads** — individual heads can show very different patterns (some focus on syntax, some on sentence boundaries, some on semantics). Averaging produces a smoothed view of overall focus.
- This is the **final encoder layer only** — earlier layers tend to handle syntax and adjacency, later layers handle task-relevant semantics. The final layer is the most relevant for interpreting the classification decision.
- Attention shows **what the model looked at**, not necessarily what drove the prediction. Strong attention to intent-relevant words is corroborating evidence, not proof of causation.

# Data reduction analysis

The deep learning models in this project were tuned with sequence-length settings of **50 tokens for BiLSTM and GRU** and **64 tokens for BERT**. These padding lengths were chosen conservatively at the start of the project, before the model performance was known. The exploratory data analysis later showed something important: every customer message in the Bitext dataset fits within **16 words**, with a median of just **9 words** and a 95th percentile of **13 words**. This means the models have been training on inputs where the majority of each padded sequence consists of zero tokens that carry no signal.

### Why this matters

The deep learning models were tuned with sequence lengths of **50 tokens** (BiLSTM, GRU) and **64 tokens** (BERT). The EDA showed every customer message fits within **16 words**. It could mean, that most of each sequence is just zeros that carry no signal.

In a production setting where the model needs to handle high message volume, the difference between a 16-token and a 64-token sequence translates directly into:

- **Inference latency** — shorter sequences respond faster
- **Memory footprint** — shorter sequences use less GPU/CPU memory
- **Throughput** — more messages can be processed per second
- **Cloud cost** — proportionally lower bills for the same workload

The sequence lengths tested are:

| Model | Lengths tested | Production setting |
|---|---|---|
| BiLSTM | 16, 24, 32, 50 | 50 |
| GRU | 16, 24, 32, 50 | 50 |
| BERT | 16, 32, 48, 64 | 64 |


This next line of code teste how reducing the TF-IDF vocabuulary size affects regression and linear SVC performance- it compares macro F1-scores

In [ ]:
# =============================================================================
# DATA REDUCTION ANALYSIS — TF-IDF feature reduction
# Sweeping max_features to test how vocabulary size affects classical ML
# =============================================================================
section("Data reduction analysis — TF-IDF feature sweep")

from sklearn.feature_extraction.text import TfidfVectorizer

vocab_sizes = [500, 1000, 2000, 5000, 10000, 50000]
# setting different maximum vocabulary sizes to test how reducing the number of features in the TF-IDF vectorizer impacts the performance
# of the Logistic Regression and LinearSVC models, with the results stored in a list for later analysis and plotting
reduction_rows = []

# Loop through different vocabulary sizes, refit the TF-IDF vectorizer with the reduced vocabulary,
# retrain the Logistic Regression and LinearSVC models, evaluate their performance on the test set
# to see how reducing the number of features (vocabulary size) impacts the macro F1 score for each model, and store the results in a list of dictionaries for later analysis and plotting.
for max_feat in vocab_sizes:
    # Refit TF-IDF with reduced vocabulary
    tfidf_r = TfidfVectorizer( # creating a new TF-IDF vectorizer with a limited vocabulary size defined by max_feat, while keeping the same n-gram range, sublinear term frequency scaling, minimum document frequency, and accent stripping as before
        ngram_range=(1, 2),
        max_features=max_feat,
        sublinear_tf=True,
        min_df=2,
        strip_accents="unicode",
    )
    X_train_r = tfidf_r.fit_transform(X_train)
    X_test_r = tfidf_r.transform(X_test)
    actual_vocab = len(tfidf_r.vocabulary_)

    # LR
    # training a new Logistic Regression model using the reduced TF-IDF features, with the same hyperparameters as before
    # (max_iter, random_state, solver, n_jobs) to ensure a fair comparison of how the reduced vocabulary impacts performance
    lr_r = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                               solver="saga", n_jobs=-1)
    lr_r.fit(X_train_r, y_train)
    f1_lr = f1_score(y_test, lr_r.predict(X_test_r), average="macro")

    # LinearSVC
    # training a new LinearSVC model using the reduced TF-IDF features, with the same hyperparameters as before
    # (max_iter, random_state, C) to ensure a fair comparison of how the reduced vocabulary impacts performance
    svc_r = LinearSVC(max_iter=2000, random_state=RANDOM_STATE, C=1.0)
    svc_r.fit(X_train_r, y_train)
    f1_svc = f1_score(y_test, svc_r.predict(X_test_r), average="macro")

    # Storing the results for this vocabulary size in a dictionary and appending it to the reduction_rows list for later analysis and plotting
    reduction_rows.append({
        "max_features": max_feat,
        "actual_vocab": actual_vocab,
        "LR_F1": f1_lr,
        "LinearSVC_F1": f1_svc,
    })

reduction_df = pd.DataFrame(reduction_rows)
print(reduction_df.round(4).to_string(index=False))

# Plot - F1 vs vocabulary size
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=reduction_df["actual_vocab"], y=reduction_df["LR_F1"],
    mode="lines+markers", name="Logistic Regression",
    line=dict(color="#2E86AB", width=2),
))
fig.add_trace(go.Scatter(
    x=reduction_df["actual_vocab"], y=reduction_df["LinearSVC_F1"],
    mode="lines+markers", name="LinearSVC",
    line=dict(color="#D1495B", width=2),
))
fig.update_layout(
    title="Data reduction analysis — Macro F1 vs TF-IDF vocabulary size",
    title_x=0.02,
    xaxis_title="Actual vocabulary size",
    yaxis_title="Test Macro F1",
    xaxis_type="log",
)
fig.show()

**Data Reduction Analysis- BiLSTM **
This section test how redued padded sequence legth affect tuned BiLSTM perfromance and training time. iut compares macro F1-score across different sequence lengths

In [ ]:
# =============================================================================
# DATA REDUCTION ANALYSIS — BiLSTM sequence length sweep
# Tests how shorter padded sequences affect F1 and training time.
# Uses the tuned BiLSTM hyperparameters; only sequence length varies.
# =============================================================================
section("Data reduction analysis — BiLSTM sequence length")

import time, gc
import tensorflow.keras.backend as K
from tensorflow.keras import callbacks
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sequence lengths to test — chosen against EDA findings (max message = 16 words)
SEQ_LENGTHS = [16, 24, 32, 50]   # 50 = current production setting

bilstm_reduction_rows = []
_orig_max_len = MAX_LEN

for seq_len in SEQ_LENGTHS:
    # Re-pad all three splits to the trial's sequence length
    X_train_r = pad_sequences(tokenizer.texts_to_sequences(X_train),
                              maxlen=seq_len, padding="post", truncating="post")
    X_val_r   = pad_sequences(tokenizer.texts_to_sequences(X_val),
                              maxlen=seq_len, padding="post", truncating="post")
    X_test_r  = pad_sequences(tokenizer.texts_to_sequences(X_test),
                              maxlen=seq_len, padding="post", truncating="post")

    # Override MAX_LEN globally so build_bilstm_model uses the reduced length
    globals()["MAX_LEN"] = seq_len

    K.clear_session(); gc.collect()
    model = build_bilstm_model(
        lstm_units=best_config["lstm_units"],
        dense_units=best_config["dense_units"],
        lstm_dropout=best_config["lstm_dropout"],
        dense_dropout=best_config["dense_dropout"],
        spatial_dropout=best_config["spatial_dropout"],
        learning_rate=best_config["learning_rate"],
        trainable_embeddings=best_config["trainable_embeddings"],
    )

    start = time.time()
    model.fit(
        X_train_r, y_train,
        validation_data=(X_val_r, y_val),
        epochs=10, batch_size=best_config["batch_size"], verbose=0,
        callbacks=[callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True,
        )],
    )
    train_time = time.time() - start

    f1 = f1_score(
        y_test,
        model.predict(X_test_r, verbose=0).argmax(axis=1),
        average="macro",
    )

    bilstm_reduction_rows.append({
        "seq_len": seq_len,
        "test_f1": round(f1, 4),
        "train_time_s": round(train_time, 1),
    })
    print(f"BiLSTM @ seq_len = {seq_len:3d}  →  F1 = {f1:.6f}  |  train = {train_time:.1f}s")

# Restore the original sequence length
globals()["MAX_LEN"] = _orig_max_len

bilstm_reduction_df = pd.DataFrame(bilstm_reduction_rows)
print("\n" + bilstm_reduction_df.to_string(index=False))

BiLSTM sequence length Analysis
This section visualises how seqquence length impacts BiLSTM perfromance and training time. It shows the trade-off between model accuracye and computational efficiency

In [ ]:
# =============================================================================
# BiLSTM plot
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# F1 vs sequence length
axes[0].plot(bilstm_reduction_df["seq_len"], bilstm_reduction_df["test_f1"],
             marker="o", linewidth=2, color="#2E86AB")
axes[0].set_xlabel("Sequence length (tokens)")
axes[0].set_ylabel("Test Macro F1")
axes[0].set_title("BiLSTM — F1 vs sequence length")
axes[0].grid(alpha=0.3)

# Training time vs sequence length
axes[1].plot(bilstm_reduction_df["seq_len"], bilstm_reduction_df["train_time_s"],
             marker="o", linewidth=2, color="#D1495B")
axes[1].set_xlabel("Sequence length (tokens)")
axes[1].set_ylabel("Training time (s)")
axes[1].set_title("BiLSTM — training time vs sequence length")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Data Reduction Analysis- GRU Sequence length**
This section tests how reduced padded sequence length affects tuned GRU perfromance and training time.
it compares macro F1-core and training duration across different sequence lenghths

In [ ]:
# =============================================================================
# DATA REDUCTION ANALYSIS — GRU sequence length sweep
# Same methodology as BiLSTM — only sequence length varies, all other
# hyperparameters use the tuned GRU configuration.
# =============================================================================
section("Data reduction analysis — GRU sequence length")

import time, gc
import tensorflow.keras.backend as K
from tensorflow.keras import callbacks
from tensorflow.keras.preprocessing.sequence import pad_sequences

 # same settings as in the BiLSTM reduction analysis, but applied to the GRU model. We will test how different sequence lengths affect the
 # GRU's performance and training time, while keeping all other hyperparameters fixed to the best configuration found during GRU tuning.

SEQ_LENGTHS = [16, 24, 32, 50]

gru_reduction_rows = []
_orig_max_len = MAX_LEN

for seq_len in SEQ_LENGTHS:
    X_train_r = pad_sequences(tokenizer.texts_to_sequences(X_train),
                              maxlen=seq_len, padding="post", truncating="post")
    X_val_r   = pad_sequences(tokenizer.texts_to_sequences(X_val),
                              maxlen=seq_len, padding="post", truncating="post")
    X_test_r  = pad_sequences(tokenizer.texts_to_sequences(X_test),
                              maxlen=seq_len, padding="post", truncating="post")

    globals()["MAX_LEN"] = seq_len

    K.clear_session(); gc.collect()
    model = build_gru_model(
        gru_units=gru_best_config["gru_units"],
        dense_units=gru_best_config["dense_units"],
        gru_dropout=gru_best_config["gru_dropout"],
        dense_dropout=gru_best_config["dense_dropout"],
        spatial_dropout=gru_best_config["spatial_dropout"],
        learning_rate=gru_best_config["learning_rate"],
        trainable_embeddings=gru_best_config["trainable_embeddings"],
    )

    start = time.time()
    model.fit(
        X_train_r, y_train,
        validation_data=(X_val_r, y_val),
        epochs=10, batch_size=gru_best_config["batch_size"], verbose=0,
        callbacks=[callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True,
        )],
    )
    train_time = time.time() - start

    f1 = f1_score(
        y_test,
        model.predict(X_test_r, verbose=0).argmax(axis=1),
        average="macro",
    )

    gru_reduction_rows.append({
        "seq_len": seq_len,
        "test_f1": round(f1, 4),
        "train_time_s": round(train_time, 1),
    })
    print(f"GRU @ seq_len = {seq_len:3d}  →  F1 = {f1:.6f}  |  train = {train_time:.1f}s")

globals()["MAX_LEN"] = _orig_max_len

gru_reduction_df = pd.DataFrame(gru_reduction_rows)
print("\n" + gru_reduction_df.to_string(index=False))

fig = px.line(
    gru_reduction_df, x="seq_len", y="test_f1",
    markers=True,
    title="GRU — F1 vs sequence length",
    labels={"seq_len": "Sequence length (tokens)", "test_f1": "Test Macro F1"},
    color_discrete_sequence=["#2E86AB"],
)
fig.update_layout(title_x=0.02, height=400)
fig.show()

fig = px.line(
    gru_reduction_df, x="seq_len", y="train_time_s",
    markers=True,
    title="GRU — training time vs sequence length",
    labels={"seq_len": "Sequence length (tokens)", "train_time_s": "Training time (s)"},
    color_discrete_sequence=["#D1495B"],
)
fig.update_layout(title_x=0.02, height=400)
fig.show()

**GRU Sequence Length Analysis- performance vs Efficiency**
This section visaulaises how sequence length affects GRO perfromance and training time.
It hughklights the trade off between accuracy and computatonal efficiency as length varies

In [ ]:
# =============================================================================
# GRU plot
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(gru_reduction_df["seq_len"], gru_reduction_df["test_f1"],
             marker="o", linewidth=2, color="#2E86AB")
axes[0].set_xlabel("Sequence length (tokens)")
axes[0].set_ylabel("Test Macro F1")
axes[0].set_title("GRU — F1 vs sequence length")
axes[0].grid(alpha=0.3)

axes[1].plot(gru_reduction_df["seq_len"], gru_reduction_df["train_time_s"],
             marker="o", linewidth=2, color="#D1495B")
axes[1].set_xlabel("Sequence length (tokens)")
axes[1].set_ylabel("Training time (s)")
axes[1].set_title("GRU — training time vs sequence length")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# DATA REDUCTION ANALYSIS — BERT sequence length sweep
# Same methodology — only MAX_LEN_TF varies. Uses tuned BERT hyperparameters
# but reduced epochs (2 instead of 4) to keep total runtime manageable.
# =============================================================================
section("Data reduction analysis — BERT sequence length")

SEQ_LENGTHS_BERT = [16, 32, 48, 64]   # 64 = current production setting

bert_reduction_rows = []
_orig_max_len_tf = MAX_LEN_TF

for seq_len in SEQ_LENGTHS_BERT:
    globals()["MAX_LEN_TF"] = seq_len

    # Reduced-epoch config for the sweep
    sweep_config = {**bert_best_config, "epochs": 2}

    _, _, summary, train_time = train_one_bert(config=sweep_config, verbose=0)

    bert_reduction_rows.append({
        "seq_len": seq_len,
        "val_f1": round(summary["val_f1"], 4),
        "train_time_s": round(train_time, 1),
    })
    print(f"BERT @ seq_len = {seq_len:3d}  →  val F1 = {summary['val_f1']:.4f}  |  train = {train_time:.1f}s")

globals()["MAX_LEN_TF"] = _orig_max_len_tf

bert_reduction_df = pd.DataFrame(bert_reduction_rows)
print("\n" + bert_reduction_df.to_string(index=False))

fig = px.line(
    bert_reduction_df, x="seq_len", y="val_f1",
    markers=True,
    title="BERT — validation F1 vs sequence length",
    labels={"seq_len": "Sequence length (tokens)", "val_f1": "Validation Macro F1"},
    color_discrete_sequence=["#2E86AB"],
)
fig.update_layout(title_x=0.02, height=400)
fig.show()

fig = px.line(
    bert_reduction_df, x="seq_len", y="train_time_s",
    markers=True,
    title="BERT — training time vs sequence length",
    labels={"seq_len": "Sequence length (tokens)", "train_time_s": "Training time (s)"},
    color_discrete_sequence=["#D1495B"],
)
fig.update_layout(title_x=0.02, height=400)
fig.show()

In [ ]:
# =============================================================================
# BERT plot
# =============================================================================
import matplotlib.pyplot as plt

# Auto-detect which F1 column the dataframe uses
f1_col = "val_f1" if "val_f1" in bert_reduction_df.columns else "test_f1"
f1_label = "Validation Macro F1" if f1_col == "val_f1" else "Test Macro F1"

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(bert_reduction_df["seq_len"], bert_reduction_df[f1_col],
             marker="o", linewidth=2, color="#2E86AB")
axes[0].set_xlabel("Sequence length (tokens)")
axes[0].set_ylabel(f1_label)
axes[0].set_title("BERT — F1 vs sequence length")
axes[0].grid(alpha=0.3)

axes[1].plot(bert_reduction_df["seq_len"], bert_reduction_df["train_time_s"],
             marker="o", linewidth=2, color="#D1495B")
axes[1].set_xlabel("Sequence length (tokens)")
axes[1].set_ylabel("Training time (s)")
axes[1].set_title("BERT — training time vs sequence length")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Reading this reduction analysis.** Each row shows the same tuned model retrained at a reduced sequence length, with all other hyperparameters held fixed. F1 is expected to remain roughly flat across all sequence lengths because the EDA confirmed every message fits within 16 tokens- longer padding adds zeros, not signal. Training time should drop noticeably for shorter sequences, with the effect largest for BERT because self-attention scales in sequence length. The deployment implication is direct: each model could be deployed at a much shorter sequence length than it was tuned at, with negligible accuracy cost and meaningful speed-up.

In [ ]:
# =============================================================================
# REDUCTION ANALYSIS — combined view across all three deep-learning tiers
# =============================================================================
section("Reduction analysis — combined view")

bilstm_reduction_df["model"] = "BiLSTM (tuned)"
gru_reduction_df["model"]    = "GRU (tuned)"
bert_reduction_df["model"]   = "BERT (tuned)"

# BERT used val_f1 instead of test_f1 — rename for consistency
bert_reduction_df = bert_reduction_df.rename(columns={"val_f1": "test_f1"})

reduction_combined = pd.concat(
    [bilstm_reduction_df, gru_reduction_df, bert_reduction_df],
    ignore_index=True,
)

print(reduction_combined.to_string(index=False))
reduction_combined.to_csv("nb_reduction_analysis_combined.csv", index=False)

# Combined chart — one line per model
fig = px.line(
    reduction_combined, x="seq_len", y="test_f1", color="model",
    markers=True,
    title="Reduction analysis — F1 vs sequence length across all tiers",
    labels={"seq_len": "Sequence length (tokens)",
            "test_f1": "Test Macro F1",
            "model": "Model"},
    color_discrete_sequence=["#2E86AB", "#D1495B", "#F4A261"],
)
fig.update_layout(title_x=0.02, height=450)
fig.show()

In [ ]:
# =============================================================================
# REDUCTION ANALYSIS — consolidated comparison table
# Mirrors the master_df style for easy side-by-side reading
# =============================================================================
section("Reduction analysis — comparison table")

# Tag each dataframe with its model name and align column naming
bilstm_reduction_df["Model"] = "BiLSTM (tuned)"
gru_reduction_df["Model"]    = "GRU (tuned)"
bert_reduction_df["Model"]   = "BERT (tuned)"

# BERT recorded val_f1 — rename for consistent column across all tiers
bert_reduction_df_aligned = bert_reduction_df.rename(columns={"val_f1": "test_f1"})

# Concatenate
reduction_master_df = pd.concat(
    [bilstm_reduction_df, gru_reduction_df, bert_reduction_df_aligned],
    ignore_index=True,
)

# Reorder columns and rename for readability
reduction_master_df = reduction_master_df.rename(columns={
    "seq_len": "Seq length",
    "test_f1": "F1",
    "train_time_s": "Time (s)",
})
reduction_master_df = reduction_master_df[["Model", "Seq length", "F1", "Time (s)"]]

# Sort: model first, then sequence length descending (50 → 16) so each model groups together
reduction_master_df = reduction_master_df.sort_values(
    ["Model", "Seq length"], ascending=[True, False]
).reset_index(drop=True)

# Print plain version
print(reduction_master_df.to_string(index=False))

# Save to CSV
reduction_master_df.to_csv("nb_reduction_analysis_combined.csv", index=False)
print("\nSaved → nb_reduction_analysis_combined.csv")

# Styled version — gradients on F1 (Blues) and Time (Reds reversed so faster = darker)
reduction_master_df.style \
    .background_gradient(subset=["F1"], cmap="Blues") \
    .background_gradient(subset=["Time (s)"], cmap="Reds_r") \
    .format({"F1": "{:.6f}", "Time (s)": "{:.1f}"})

# Models Evaluation

Every model is evaluated on the held-out test set using the same set of metrics:

- **Accuracy** — overall proportion of correct predictions.
- **Macro precision, recall, F1** — each class contributes equally to the score, so rare classes are not hidden by common ones.
- **ROC-AUC (macro, one-vs-rest)** — measures how well the model ranks the correct class relative to the other 26, averaged across all classes.

Macro F1 is the main metric for model selection because it gives equal weight to every intent, which matches the business view that all 27 intents are equally important to classify correctly.

The following visualisations support the evaluation:

- **Training / validation loss curves** — to check that each model converged and did not overfit.
- **Confusion matrices** — to see which intent pairs are most often confused with each other.
- **Per-intent F1 scores** — to identify which classes are easy and which are hard for each model.
- **Attention maps (transformers only)** — to show which words the transformer focused on when making its prediction.

LinearSVC has no native probability output, so its `decision_function` scores are passed through softmax inside the shared evaluation helper before ROC-AUC is computed. This is the only place model-specific evaluation logic is needed.

## Cross-Model Consolidation

All trained models are gathered into a single comparison structure here. This avoids renaming variables across the notebook and gives one place to:

1. compare metrics side-by-side
2. plot a unified comparison chart
3. plot confusion matrices for every model

In [ ]:
# =============================================================================
# CROSS-MODEL CONSOLIDATION
# =============================================================================
section("Cross-model consolidation")

# Create a dictionary called `model_registry`
# Each key is a model name, and each value is another dictionary
# containing that model's predictions, probabilities, metrics, and model category
model_registry = {
    "Logistic Regression": {
        "y_pred":  globals().get("y_pred_lr"),
        "y_proba": globals().get("y_proba_lr"),
        "metrics": globals().get("lr_metrics"),
        "tier":    "Classical ML",
    },
    "LR (tuned)": {
        "y_pred":  globals().get("y_pred_lr_tuned"),
        "y_proba": globals().get("y_proba_lr_tuned"),
        "metrics": globals().get("lr_tuned_metrics"),
        "tier":    "Classical ML",
    },
    "LinearSVC": {
        "y_pred":  globals().get("y_pred_svc"),
        "y_proba": globals().get("y_proba_svc"),
        "metrics": globals().get("svc_metrics"),
        "tier":    "Classical ML",
    },
    "LinearSVC (tuned)": {
        "y_pred":  globals().get("y_pred_svc_tuned"),
        "y_proba": globals().get("y_proba_svc_tuned"),
        "metrics": globals().get("svc_tuned_metrics"),
        "tier":    "Classical ML",
    },
    "BiLSTM (baseline)": {
        "y_pred":  globals().get("baseline_pred"),
        "y_proba": globals().get("baseline_proba"),
        "metrics": globals().get("baseline_metrics"),
        "tier":    "Deep Learning",
    },
    "BiLSTM (tuned)": {
        "y_pred":  globals().get("tuned_pred"),
        "y_proba": globals().get("tuned_proba"),
        "metrics": globals().get("tuned_metrics"),
        "tier":    "Deep Learning",
    },
    "GRU (baseline)": {
        "y_pred":  globals().get("gru_baseline_pred"),
        "y_proba": globals().get("gru_baseline_proba"),
        "metrics": globals().get("gru_baseline_metrics"),
        "tier":    "Deep Learning",
    },
    "GRU (tuned)": {
        "y_pred":  globals().get("gru_tuned_pred"),
        "y_proba": globals().get("gru_tuned_proba"),
        "metrics": globals().get("gru_tuned_metrics"),
        "tier":    "Deep Learning",
    },
    "BERT-base (tuned)": {
        "y_pred":  globals().get("bert_pred"),
        "y_proba": globals().get("bert_proba"),
        "metrics": globals().get("bert_metrics"),
        "tier":    "Transformer",
    },
    "BERT (baseline)": {
        "y_pred":  globals().get("bert_baseline_pred"),
        "y_proba": globals().get("bert_baseline_proba"),
        "metrics": globals().get("bert_baseline_metrics"),
        "tier":    "Transformer",
    },
}

# Create a filtered dictionary called `trained`
# Keep only the models whose predictions exist
# This avoids including models that were not actually trained or evaluated
trained = {k: v for k, v in model_registry.items() if v["y_pred"] is not None}
print(f"Models in comparison: {len(trained)}")
for m in trained:
    print(f"  ✓ {m}")

rows = []
for name, info in trained.items(): # iterate through the trained models and their information in the `trained` dictionary
    m = info["metrics"] or {}
    rows.append({
        "Model":     name,
        "Tier":      info["tier"],
        "Accuracy":  m.get("accuracy"),
        "Precision": m.get("precision"),
        "Recall":    m.get("recall"),
        "F1":        m.get("f1"),
        "ROC-AUC":   m.get("roc_auc"),
        "Time (s)":  m.get("time_s") or m.get("Time(s)"),
    })

master_df = (
    pd.DataFrame(rows)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

print("\n" + master_df.round(4).to_string(index=False))
master_df.to_csv("nb_all_models_results.csv", index=False)
print("\nSaved → nb_all_models_results.csv  (used by NB4)")

master_df.style \
    .background_gradient(subset=["F1"],        cmap="Blues") \
    .background_gradient(subset=["ROC-AUC"],   cmap="Greens") \
    .background_gradient(subset=["Precision"], cmap="Oranges") \
    .format({c: "{:.6f}" for c in ["Accuracy","Precision","Recall","F1","ROC-AUC"]}) \
    .format({"Time (s)": "{:.1f}"})

## Output Explanation
The cross-model consolidation table compares the performance of all trained models across the main evaluation metrics. All models achieved very strong results, with F1 scores above 0.99. Among them, the tuned BiLSTM achieved the highest overall performance, followed closely by the BERT and tuned GRU models. LinearSVC also performed competitively despite being a classical machine learning approach. This comparison shows that while all approaches were effective for customer-intent classification, the tuned BiLSTM provided the best balance of predictive performance in this study.

In [ ]:
# =============================================================================
# COMPARISON CHART — grouped bar of all key metrics
# =============================================================================
section("Comparison chart — all models")

metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]

# Convert the DataFrame from wide format to long format
# Before melting:
#   one row = one model, and each metric is in a separate column
# After melting:
#   one row = one model-metric pair
#
# id_vars=["Model", "Tier"]
#   Keep the Model name and Tier unchanged
#
# value_vars=metric_cols
#   Melt only these metric columns: Accuracy, Precision, Recall, F1, ROC-AUC
#
# var_name="Metric"
#   Store the metric name (e.g., Accuracy, F1) in a new column called "Metric"
#
# value_name="Score"
#   Store the corresponding metric value in a new column called "Score"

plot_df = master_df.melt(
    id_vars=["Model", "Tier"],
    value_vars=metric_cols,
    var_name="Metric",
    value_name="Score",
)

fig = px.bar(
    plot_df,
    x="Model", y="Score", color="Metric",
    barmode="group",
    title="Cross-tier model comparison — all metrics",
    color_discrete_sequence=["#2E86AB", "#D1495B", "#F4A261", "#2A9D8F", "#6A4C93"],
)
fig.update_layout(
    title_x=0.02,
    yaxis=dict(title="Score", range=[0, 1.0]),
    xaxis_title=None,
    legend_title_text="Metric",
    height=500,
)
fig.show()

### Summary of the output

The cross-tier comparison confirms a clear tier hierarchy, though the margins between
tiers are narrower than expected given the difficulty of the task.

**Transformer tier (best).** BERT-base (tuned, lr = 3e-5) achieved the highest test
macro F1 of **0.9965**, with BERT-base (baseline) close behind at 0.9955. Both reach
ROC-AUC = 1.0000. This answers **RQ3**: transformers are the strongest tier, but the
payoff over classical ML is modest (~1.5 pp F1) rather than dramatic — suggesting the
dataset is sufficiently separable by surface vocabulary that even TF-IDF captures most
of the signal.

**Deep learning tier (second).** GRU (tuned) is the surprise result: F1 = **0.9958**,
placing it above both BERT-base (baseline) and all classical models. BiLSTM (tuned)
reaches 0.9946. Fine-tuned embeddings outperform frozen embeddings in both recurrent
models, confirming that GloVe domain adaptation is worth the extra training cost
(answers **RQ2**).

**Classical ML tier (baseline).** LinearSVC (untuned) achieves F1 = **0.9950** — only
0.0015 below the best model. This is the answer to **RQ1**: the classical ceiling is
surprisingly high. TF-IDF + LinearSVC captures the majority of intent signal from
unigrams and bigrams alone, which is consistent with the EDA finding that most intents
have partially distinct vocabularies. The remaining errors concentrate on the
cancel_order / change_order pair, as predicted.

**Error structure (RQ4).** Across all tiers, the hardest intents are consistently
`change_order` and `cancel_order`, with `track_order` and `check_invoice` / `get_invoice`
forming a secondary confusion cluster. Cross-category errors are rare in every model,
confirming that confusion is driven by within-category vocabulary overlap rather than
random noise.

In [ ]:
# =============================================================================
# F1 BY TIER — quick view of best model per family
# =============================================================================
section("Best F1 per tier")

best_per_tier = (
    master_df.sort_values("F1", ascending=False)
    .groupby("Tier", as_index=False).first()
)

fig = px.bar(
    best_per_tier.sort_values("F1"),
    x="F1", y="Tier", color="Model",
    orientation="h",
    title="Best macro F1 per tier",
    text="Model",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(title_x=0.02, xaxis=dict(range=[0, 1]), height=350,
                  showlegend=False)
fig.update_traces(textposition="inside")
fig.show()

### Best model per tier — output explanation

The chart shows the single best-performing model from each of the three tiers, selected
by test macro F1.

| Tier | Best model | Test macro F1 |
|------|-----------|---------------|
| Transformer | BERT-base (tuned, lr = 3e-5) | 0.9965 |
| Deep Learning | GRU (tuned, fine-tuned embeddings) | 0.9958 |
| Classical ML | LinearSVC (untuned, C = 1) | 0.9950 |

The gap between tiers is narrow - only 0.0015 F1 separates the best transformer from
the best classical model. This reflects the nature of the dataset: synthetic, balanced,
and short-text, where TF-IDF bigrams already capture most of the discriminative signal.
The tier ordering is nonetheless consistent with expectations — contextual models
outperform bag-of-words models, and self-attention outperforms sequential hidden states.

Notably, the best classical model (LinearSVC untuned, C=1) outperforms its tuned
version (C=3) by a negligible margin, and both sit below the tuned GRU — confirming
that for this task, the representation matters more than hyperparameter precision within
a tier.

In [ ]:
# =============================================================================
# ROC CURVES — all trained models (log-scale FPR for visual separation)
# =============================================================================
section("ROC curves — all trained models")

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from scipy.special import softmax as _softmax

# One-hot encode y_test for one-vs-rest ROC
y_test_bin = label_binarize(y_test, classes=range(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(9, 7))
# Loop through each trained model and plot its ROC curve.
# np .asarray() ensures we have a NumPy array for the probabilities, which is needed for the ROC calculations.
# trained is a dictionary where each key is a model name and each value is another dictionary containing that model's predictions, probabilities, metrics, and tier.
for name, info in trained.items():
    y_proba = np.asarray(info["y_proba"])

    # LinearSVC fix — convert decision scores to probabilities
    if not np.allclose(y_proba.sum(axis=1), 1.0, atol=1e-3):
        y_proba = _softmax(y_proba, axis=1)

    # Micro-average ROC across all 27 classes
    fpr, tpr, _ = roc_curve(y_test_bin.ravel(), y_proba.ravel())
    ax.plot(fpr, tpr, linewidth=1.8, label=f"{name} (AUC = {auc(fpr, tpr):.6f})")

# Diagonal reference — random classifier
ax.plot([1e-4, 1], [1e-4, 1], "k--", linewidth=1, alpha=0.5)

# Log-scale FPR + zoomed TPR — separates the curves where they actually differ
ax.set_xscale("log")
ax.set_xlim([1e-4, 1.0])
ax.set_ylim([0.95, 1.005])

ax.set_xlabel("False Positive Rate (log scale)")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — all models (micro-average across 27 classes)")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

### ROC curves — output explanation

All models achieve micro-averaged AUC > 0.9999, so the x-axis uses a log scale and
the y-axis is zoomed to 0.95–1.0 to make the differences between curves visible. On
a standard linear scale every line would collapse into the top-left corner. The curves
confirm the macro F1 ranking but add little new information at this performance level,
which is why macro F1 remains the primary selection metric.

The next line of code compares all models using precision recall curves. it compares theirperformance across all classes using micro-averaging.

In [ ]:
# =============================================================================
# PRECISION-RECALL CURVES — all trained models
# =============================================================================
section("Precision-recall curves — all trained models")

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
from scipy.special import softmax as _softmax

# One-hot encode y_test for one-vs-rest PR curves
y_test_bin = label_binarize(y_test, classes=range(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(9, 7))

for name, info in trained.items():
    y_proba = np.asarray(info["y_proba"])

    # LinearSVC fix — convert decision scores to probabilities
    if not np.allclose(y_proba.sum(axis=1), 1.0, atol=1e-3):
        y_proba = _softmax(y_proba, axis=1)

    # Micro-average PR across all 27 classes
    precision, recall, _ = precision_recall_curve(
        y_test_bin.ravel(), y_proba.ravel()
    )
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, linewidth=1.8,
            label=f"{name} (AP = {pr_auc:.4f})")

# Less aggressive zoom — keeps the full curve visible
ax.set_xlim([0.8, 1.005])
ax.set_ylim([0.8, 1.005])

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — all models (zoomed)")
ax.legend(loc="lower left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Precision-recall curves - output explanation

All models score above 0.99 average precision, so the axes are zoomed to 0.8–1.0 to
make the small differences between curves visible. The tier ranking matches the macro
F1 results - transformers sit highest, followed by deep learning, then classical ML.

# **Results Summary**

This notebook implemented and evaluated model configurations across three tiers for
27-class customer support intent classification on the Bitext dataset (26,872 samples).

**Performance ranking (test macro F1):**

| Rank | Model | Tier | Macro F1 |
|------|-------|------|----------|
| 1 | BERT-base (tuned, lr=3e-5) | Transformer | 0.9965 |
| 2 | GRU (tuned, fine-tuned embeddings) | Deep Learning | 0.9958 |
| 3 | BERT-base (baseline, lr=2e-5) | Transformer | 0.9955 |
| 4 | LinearSVC (untuned, C=1) | Classical ML | 0.9950 |
| 5 | LinearSVC (tuned, C=3) | Classical ML | 0.9946 |
| 5 | BiLSTM (tuned, fine-tuned embeddings) | Deep Learning | 0.9946 |
| 7 | LR (tuned, C=100) | Classical ML | 0.9943 |
| 8 | LR (untuned) | Classical ML | 0.9941 |
| 9 | BiLSTM (baseline, frozen embeddings) | Deep Learning | 0.9916 |
| 10 | GRU (baseline, frozen embeddings) | Deep Learning | 0.9906 |

**Key findings:**

- The classical ceiling (RQ1) is high: LinearSVC reaches 0.9950 on TF-IDF alone,
  leaving only ~1.5 pp of headroom for deeper architectures.
- Fine-tuned GloVe embeddings (RQ2) consistently outperform frozen embeddings by
  ~0.4–0.5 pp F1 in both BiLSTM and GRU, confirming that domain adaptation matters
  even on short utterances.
- Transformers (RQ3) are the strongest tier, but the margin over the best classical
  model is narrow (0.0015 F1) — the dataset's synthetic balance and limited vocabulary
  overlap mean TF-IDF already captures most discriminative signal.
- Error patterns (RQ4) are consistent across all tiers: `cancel_order` vs
  `change_order` is the hardest pair in every model, and cross-category errors are rare.

**Deployment recommendation:** LinearSVC is the practical choice for low-latency
production routing — it is within 0.0015 F1 of the best transformer, trains in seconds,
and requires no GPU at inference time. BERT-base (tuned) is the correct choice where
accuracy is the primary constraint and inference latency is acceptable.

## Model Selection — Trade-offs, Deployment Context, and Why Macro F1 Is the Right Metric

---

### Traditional ML vs Deep Learning vs Transformers

| | Classical ML (LR, LinearSVC) | Deep Learning (BiLSTM, GRU) | Transformers (BERT) |
|---|---|---|---|
| **Training time** | Seconds | 1–2 minutes | 10–15 minutes |
| **Inference speed** | ~0.1 ms per query | ~5 ms per query | ~50 ms per query |
| **GPU required** | No | Optional | Strongly recommended |
| **Interpretability** | High — feature weights directly inspectable | Low | Very low |
| **Short text handling** | Good with bigrams | Good with GloVe | Excellent — WordPiece |
| **Typo robustness** | Poor — OOV words become zeros | Moderate — OOV mapped to GloVe zero vector | Good — WordPiece splits unknowns into subword pieces |
| **Data requirements** | Low — works well with ~500 examples per class | Moderate — benefits from >1,000 per class | Low — pretrained weights transfer well from small data |
| **Vocabulary overlap problem** | Struggles — cancel_order and change_order share TF-IDF features | Better — embeddings encode semantic distance | Best — attention weights can distinguish word roles in context |

**Benefits of Classical ML vs Deep Learnign ML:**
Classical ML is the practical deployment choice — LinearSVC reaches F1 = 0.9950 in
seconds with no GPU and fully inspectable feature weights, but it cannot resolve
vocabulary overlap between semantically close intents (cancel_order vs change_order)
because TF-IDF has no concept of word order or meaning. Deep learning with fine-tuned
GloVe embeddings closes part of that gap (+0.0008 F1 for GRU tuned) by encoding
sequential context, though the gain is modest on 8-word utterances where there is little
sequence to model. Transformers achieve the highest F1 (0.9965) by allowing every token
to attend directly to every other token and handling typos gracefully via WordPiece
subword tokenisation, but the payoff over LinearSVC is only 0.0015 F1 — on this
balanced synthetic dataset the transformer is solving a vocabulary overlap problem that
largely does not exist at scale.



## **Research Questions: Findings**

**RQ1:How well can a simple linear classifier with TF-IDF distinguish 27 overlapping classes?**

The Classical machine learning models (logistics Regression and Linear SVC) had very strong performance. The best classical model (Linear SVC) achieved a macro F1 score of 0.995. this is a very strong baseline depsite the presence of overlapping variables across 27 intent classes. TD-IDF worked weel at distinguishing keywords and patterns.

The remaining errors were mainly intents that are semantically similar in that TF-IDF could not capture the contextual meaning. This means that classical models work well when the intent is keyword driven but do not work well when intent is driven by context.
one can argue that a near perfect dataset (clean synthetic dataset with short phrase that is evenly distributed) is a major contribution to this high result.

**RQ2: Recurrent uplift. Do sequence models (BiLSTM, GRU) with pretrained GloVe embeddings give a meaningful improvement over the classical baseline on messages this short?**

RNN models (BiLSTM and GRU) differ from calssical ML models in that they capture sequential dependencies and contextual relationships between words using pretrained GloVe embeddings

The best RNN model (GR with fine tuned embeddings) achived a macro F1 code of 0.9958 (a slight improvement from the 0.995 ahceived by linear SVC).

again, the type of datset might be a contributor to this limited improvement:

Short text inputs mean the intent in this datset is largely keyword driven rather and this reduces the benefit of sequence modelling intent classification. pretrained GlOve embeddings add some semantic understanding but not enough to make a dramatice improvement in performance.

Overal GRU and BiLSTM provide measurable imporovements but not much. This only mean that sequential modelling may not be necessary for this particular task.

**RQ3:The data includes different writing styles. The Bitext dataset labels each message with flags for things like misspellings, colloquial phrasing, and politeness. This means the data covers the kind of variation real customers produce, rather than only clean, well-formed sentences.**

BERT (a transformer model) has the best oerall poerfromance in this case study- achieving a macro F1 score of .9965.  BUt the perfromance difference between BERT and the classical baseline is approx. 1.5% point, and only 0.07% points over the GRU model. Although BERT shows superior contextual understanding, the level of improvement is relatively small copmpared to the higher computation cost.

to translate this physically, training time inceases from mere seconds  (Classical ML) to miutes (BERT) this mean substantial higher hadware reuiremenst (GPU/CPU).

so why BERT s tyhe best performer, the results show that its perfrmoance imporvement is not substantial for this dataset.